# Línea A: preparación de datos, representaciones y modelos KNN

Este notebook desarrolla el flujo completo de la Línea A: carga y preparación de comentarios abiertos, generación de variantes de preprocesamiento, construcción de representaciones TF-IDF y Word2Vec, etiquetado manual, entrenamiento KNN, ajuste de hiperparámetros y calibración de umbrales.

## Etapa 1: Carga e inspección inicial

Se importan las librerías, se carga la base original y se revisan su estructura, dimensiones, valores nulos y columnas de comentarios disponibles.

In [1]:
import pandas as pd
import numpy as np
import re
import unicodedata
from html import unescape
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer

try:
    from nltk.stem.snowball import SnowballStemmer
    stemmer_es = SnowballStemmer("spanish")
except Exception:
    stemmer_es = None

try:
    import spacy
    nlp = spacy.load("es_core_news_sm")
except Exception:
    nlp = None

df = pd.read_csv("Datos_Total.csv", sep=";", engine="python", skiprows=[1])

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

In [2]:
df.shape

(13113, 233)

In [3]:
print(df["Q2_10_TEXT"].isna().value_counts())
print(df["Q3"].isna().value_counts())
print(df["Q4"].isna().value_counts())
print(df["Q5"].isna().value_counts())

Q2_10_TEXT
True     11157
False     1956
Name: count, dtype: int64
Q3
True     11479
False     1634
Name: count, dtype: int64
Q4
True     12774
False      339
Name: count, dtype: int64
Q5
True     10923
False     2190
Name: count, dtype: int64


In [4]:
COLUMNAS_COMENTARIOS = ["Q2_10_TEXT","Q3","Q4","Q5"]

for col in COLUMNAS_COMENTARIOS:
    if col not in df.columns:
        print(f"Advertencia no existe la columna: {col}")

print("Columnas de comentarios encontradas:")
print(COLUMNAS_COMENTARIOS)
        

Columnas de comentarios encontradas:
['Q2_10_TEXT', 'Q3', 'Q4', 'Q5']


In [5]:
df

,StartDate,EndDate,Status,IPAddress,Progress,Duration (in seconds),Finished,RecordedDate,ResponseId,RecipientLastName,RecipientFirstName,RecipientEmail,ExternalReference,LocationLatitude,LocationLongitude,DistributionChannel,UserLanguage,Q1,Q2,Q2_10_TEXT,Q3,Q4,Q5,Q6,Q6_9_TEXT,Q7_1,Q7_2,Q7_3,Q7_4,Q7_5,Q8,Q9,Q9_6_TEXT,Q10,Q10_4_TEXT,Q11,Q12,Q13,Q14,Q15,ENCUESTA,WHATSAPP,FECHA_PAGO,OSU,CODIGO_SAP,APELLIDO_MATERNO,RUT_EMPRESA,DV_EMPRESA,RAZON_SOCIAL,CENTRO_ATENCION_PAGO,INICIO_BENEFICIO,TIPIFICACION,ESTADO_CALIFICACION,TIPO_PAGO,ESTADO_PAGO,PAGO_BBEE,SINIESTRO,CONTRATO,RUT_TRABAJADOR,DV_TRABAJADOR,INICIO_PAGO_OSU,TIPIFICACION_VF,ResponseID,Q14_4_TEXT - Effort,Q14_4_TEXT - Emotion,Q14_4_TEXT - Sentiment Polarity,Q14_4_TEXT - Sentiment Score,Q14_4_TEXT - Topics,Q14_4_TEXT - Parent Topics,Q14_4_TEXT - Sentiment,Q14_4_TEXT - Actionability,Q14_4_TEXT - Effort Numeric,Q14_4_TEXT - Emotion Intensity,Q14_4_TEXT - Topic Sentiment Label,Q14_4_TEXT - Topic Sentiment Score,Q15_4_TEXT - Effort,Q15_4_TEXT - Emotion,Q15_4_TEXT - Sentiment Polarity,Q15_4_TEXT - Sentiment Score,Q15_4_TEXT - Topics,Q15_4_TEXT - Parent Topics,Q15_4_TEXT - Sentiment,Q15_4_TEXT - Actionability,Q15_4_TEXT - Effort Numeric,Q15_4_TEXT - Emotion Intensity,Q15_4_TEXT - Topic Sentiment Label,Q15_4_TEXT - Topic Sentiment Score,Q16_3_TEXT - Effort,Q16_3_TEXT - Emotion,Q16_3_TEXT - Sentiment Polarity,Q16_3_TEXT - Sentiment Score,Q16_3_TEXT - Topics,Q16_3_TEXT - Parent Topics,Q16_3_TEXT - Sentiment,Q16_3_TEXT - Actionability,Q16_3_TEXT - Effort Numeric,Q16_3_TEXT - Emotion Intensity,Q16_3_TEXT - Topic Sentiment Label,Q16_3_TEXT - Topic Sentiment Score,Q17_3_TEXT - Effort,Q17_3_TEXT - Emotion,Q17_3_TEXT - Sentiment Polarity,Q17_3_TEXT - Sentiment Score,Q17_3_TEXT - Topics,Q17_3_TEXT - Parent Topics,Q17_3_TEXT - Sentiment,Q17_3_TEXT - Actionability,Q17_3_TEXT - Effort Numeric,Q17_3_TEXT - Emotion Intensity,Q17_3_TEXT - Topic Sentiment Label,Q17_3_TEXT - Topic Sentiment Score,Q18_3_TEXT - Effort,Q18_3_TEXT - Emotion,Q18_3_TEXT - Sentiment Polarity,Q18_3_TEXT - Sentiment Score,Q18_3_TEXT - Topics,Q18_3_TEXT - Parent Topics,Q18_3_TEXT - Sentiment,Q18_3_TEXT - Actionability,Q18_3_TEXT - Effort Numeric,Q18_3_TEXT - Emotion Intensity,Q18_3_TEXT - Topic Sentiment Label,Q18_3_TEXT - Topic Sentiment Score,Q19_3_TEXT - Effort,Q19_3_TEXT - Emotion,Q19_3_TEXT - Sentiment Polarity,Q19_3_TEXT - Sentiment Score,Q19_3_TEXT - Topics,Q19_3_TEXT - Parent Topics,Q19_3_TEXT - Sentiment,Q19_3_TEXT - Actionability,Q19_3_TEXT - Effort Numeric,Q19_3_TEXT - Emotion Intensity,Q19_3_TEXT - Topic Sentiment Label,Q19_3_TEXT - Topic Sentiment Score,Q2_11_TEXT - Actionability,Q2_11_TEXT - Effort,Q2_11_TEXT - Effort Numeric,Q2_11_TEXT - Emotion Intensity,Q2_11_TEXT - Emotion,Q2_11_TEXT - Parent Topics,Q2_11_TEXT - Sentiment Polarity,Q2_11_TEXT - Sentiment Score,Q2_11_TEXT - Sentiment,Q2_11_TEXT - Topic Sentiment Label,Q2_11_TEXT - Topic Sentiment Score,Q2_11_TEXT - Topics,Q4_9_TEXT - Actionability,Q4_9_TEXT - Effort,Q4_9_TEXT - Effort Numeric,Q4_9_TEXT - Emotion Intensity,Q4_9_TEXT - Emotion,Q4_9_TEXT - Parent Topics,Q4_9_TEXT - Sentiment Polarity,Q4_9_TEXT - Sentiment Score,Q4_9_TEXT - Sentiment,Q4_9_TEXT - Topic Sentiment Label,Q4_9_TEXT - Topic Sentiment Score,Q4_9_TEXT - Topics,Q8_4_TEXT - Actionability,Q8_4_TEXT - Effort,Q8_4_TEXT - Effort Numeric,Q8_4_TEXT - Emotion Intensity,Q8_4_TEXT - Emotion,Q8_4_TEXT - Parent Topics,Q8_4_TEXT - Sentiment Polarity,Q8_4_TEXT - Sentiment Score,Q8_4_TEXT - Sentiment,Q8_4_TEXT - Topic Sentiment Label,Q8_4_TEXT - Topic Sentiment Score,Q8_4_TEXT - Topics,Q5_9_TEXT - Actionability,Q5_9_TEXT - Effort,Q5_9_TEXT - Effort Numeric,Q5_9_TEXT - Emotion Intensity,Q5_9_TEXT - Emotion,Q5_9_TEXT - Parent Topics,Q5_9_TEXT - Sentiment Polarity,Q5_9_TEXT - Sentiment Score,Q5_9_TEXT - Sentiment,Q5_9_TEXT - Topic Sentiment Label,Q5_9_TEXT - Topic Sentiment Score,Q5_9_TEXT - Topics,Q13_8_TEXT - Actionability,Q13_8_TEXT - Effort,Q13_8_TEXT - Effort Numeric,Q13_8_TEXT - Emotion Intensity,Q13_8_

## Etapa 2: Estructuración de comentarios

Las preguntas abiertas se transforman desde formato ancho a formato largo mediante `melt`, generando una fila por comentario. Posteriormente, se eliminan registros nulos, vacíos o residuales para conservar únicamente comentarios válidos.

In [6]:
df = df.copy()
df["ResponseId"] = df.index

df_comentarios = df.melt(
    id_vars=["ResponseId"],
    value_vars=COLUMNAS_COMENTARIOS,
    var_name="pregunta_origen",
    value_name="comentario_original"
)

df_comentarios

,ResponseId,pregunta_origen,comentario_original
0,0,Q2_10_TEXT,NaN
1,1,Q2_10_TEXT,NaN
2,2,Q2_10_TEXT,NaN
3,3,Q2_10_TEXT,NaN
4,4,Q2_10_TEXT,NaN
...,...,...,...
52447,13108,Q5,NaN
52448,13109,Q5,NaN
52449,13110,Q5,NaN
52450,13111,Q5,NaN


In [7]:
df_comentarios["comentario_original"] = df_comentarios["comentario_original"].astype("string").str.strip() 

df_comentarios = df_comentarios[df_comentarios["comentario_original"].notna() & df_comentarios["comentario_original"].ne("")& df_comentarios["comentario_original"].str.lower().ne("nan")].copy()

df_comentarios = df_comentarios.reset_index(drop=True)

print("Candidad de comentarios validos: ",len(df_comentarios))
display(df_comentarios.head(10))

Candidad de comentarios validos:  6119


,ResponseId,pregunta_origen,comentario_original
0,29,Q2_10_TEXT,En qué me hicieron 2 trasferencias de pago no sale si es el pago de licencia médica no informaron el monto ni nada
1,35,Q2_10_TEXT,Debería el monto ser tu sueldo base
2,39,Q2_10_TEXT,"Recibí el dinero, pero no me llegó ningún correo o mensaje del respectivo pago"
3,73,Q2_10_TEXT,La incertidumbre del día de fecha de pago ya que la página no me permite el acceso al portal online
4,83,Q2_10_TEXT,Regular
5,91,Q2_10_TEXT,Muy poco el pago con la referencia que gana diariamente uno
6,132,Q2_10_TEXT,En la mutual me dicen que tengo el pago aprobado para cobrar en BancoEstado y en banco estado me dicen que no hay ningún pago a mi nombre hasta ahora solo he cobrado una de 3 licencias
7,134,Q2_10_TEXT,Sigo esperando los pagos ya me dijeron que x fecha pagarian pero tuve yo que preguntar porque jamás dan información y de paso me contabilizaron mal el primer pago en fin no me gusto
8,145,Q2_10_TEXT,consulte en mutual x unos dias de reposo y quedaron de pagar el 1 de abril y no he tenido respuesta son 21 días de reposo y solo he recibido pago de 17 días
9,157,Q2_10_TEXT,Listo


## Etapa 3: Limpieza y normalización inicial

Se aplica una limpieza conservadora a los comentarios, incluyendo normalización Unicode, eliminación de ruido técnico y expansión de abreviaciones. Luego, se crea una tokenización simple que servirá como base para las variantes posteriores.

In [8]:
PATRON_HTML = re.compile(r"<[^>]+>")
PATRON_URL = re.compile(r"(https?://\S+|www\.\S+)", re.IGNORECASE)
PATRON_EMAIL = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w+\b", re.IGNORECASE)
PATRON_TELEFONO = re.compile(r"(\+?56)?\s?9\s?\d{4}\s?\d{4}")
PATRON_NUMERO_LARGO = re.compile(r"\b\d{5,}\b")

def normalizar_unicode(texto):
    """Normaliza texto Unicode y entidades HTML."""
    if pd.isna(texto):
        return ""

    texto = str(texto)
    texto = unescape(texto)
    texto = unicodedata.normalize("NFKC", texto)

    return texto

def limpiar_ruido_basico(texto):
    """Aplica limpieza conservadora al comentario."""

    texto = normalizar_unicode(texto)

    texto = texto.lower()

    texto = PATRON_HTML.sub(" ", texto)

    texto = PATRON_URL.sub(" url ", texto)
    texto = PATRON_EMAIL.sub(" email ", texto)
    texto = PATRON_TELEFONO.sub(" telefono ", texto)
    texto = PATRON_NUMERO_LARGO.sub(" numero ", texto)

    texto = texto.replace("\n", " ").replace("\r", " ").replace("\t", " ")

    texto = re.sub(r"(.)\1{2,}", r"\1\1", texto)

    texto = re.sub(r"[^a-záéíóúüñ0-9\s]", " ", texto)

    texto = re.sub(r"\s+", " ", texto).strip()

    return texto

df_comentarios["comentario_limpio"] = df_comentarios["comentario_original"].apply(limpiar_ruido_basico)

display(df_comentarios[["comentario_original", "comentario_limpio"]].head(20))

,comentario_original,comentario_limpio
0,En qué me hicieron 2 trasferencias de pago no sale si es el pago de licencia médica no informaron el monto ni nada,en qué me hicieron 2 trasferencias de pago no sale si es el pago de licencia médica no informaron el monto ni nada
1,Debería el monto ser tu sueldo base,debería el monto ser tu sueldo base
2,"Recibí el dinero, pero no me llegó ningún correo o mensaje del respectivo pago",recibí el dinero pero no me llegó ningún correo o mensaje del respectivo pago
3,La incertidumbre del día de fecha de pago ya que la página no me permite el acceso al portal online,la incertidumbre del día de fecha de pago ya que la página no me permite el acceso al portal online
4,Regular,regular
5,Muy poco el pago con la referencia que gana diariamente uno,muy poco el pago con la referencia que gana diariamente uno
6,En la mutual me dicen que tengo el pago aprobado para cobrar en BancoEstado y en banco estado me dicen que no hay ningún pago a mi nombre hasta ahora solo he cobrado una de 3 licencias,en la mutual me dicen que tengo el pago aprobado para cobrar en bancoestado y en banco estado me dicen que no hay ningún pago a mi nombre hasta ahora solo he cobrado una de 3 licencias
7,Sigo esperando los pagos ya me dijeron que x fecha pagarian pero tuve yo que preguntar porque jamás dan información y de paso me contabilizaron mal el primer pago en fin no me gusto,sigo esperando los pagos ya me dijeron que x fecha pagarian pero tuve yo que preguntar porque jamás dan información y de paso me contabilizaron mal el primer pago en fin no me gusto
8,consulte en mutual x unos dias de reposo y quedaron de pagar el 1 de abril y no he tenido respuesta son 21 días de reposo y solo he recibido pago de 17 días,consulte en mutual x unos dias de reposo y quedaron de pagar el 1 de abril y no he tenido respuesta son 21 días de reposo y solo he recibido pago de 17 días
9,Listo,listo


In [9]:
DICCIONARIO_ABREVIACIONES = {
    "pq": "porque",
    "xq": "porque",
    "xk": "porque",
    "q": "que",
    "tb": "también",
    "tbn": "también",
    "hrs": "horas",
    "hr": "hora",
    "nro": "número",
    "num": "número",
    "aprox": "aproximadamente"
}

def expandir_abreviaciones(texto):
    """Expande abreviaciones frecuentes."""

    for abreviacion, reemplazo in DICCIONARIO_ABREVIACIONES.items():
        patron = r"\b" + re.escape(abreviacion) + r"\b"
        texto = re.sub(patron, reemplazo, texto)

    return texto

df_comentarios["comentario_limpio"] = df_comentarios["comentario_limpio"].apply(expandir_abreviaciones)

display(df_comentarios[["comentario_original", "comentario_limpio"]].head(20))

,comentario_original,comentario_limpio
0,En qué me hicieron 2 trasferencias de pago no sale si es el pago de licencia médica no informaron el monto ni nada,en qué me hicieron 2 trasferencias de pago no sale si es el pago de licencia médica no informaron el monto ni nada
1,Debería el monto ser tu sueldo base,debería el monto ser tu sueldo base
2,"Recibí el dinero, pero no me llegó ningún correo o mensaje del respectivo pago",recibí el dinero pero no me llegó ningún correo o mensaje del respectivo pago
3,La incertidumbre del día de fecha de pago ya que la página no me permite el acceso al portal online,la incertidumbre del día de fecha de pago ya que la página no me permite el acceso al portal online
4,Regular,regular
5,Muy poco el pago con la referencia que gana diariamente uno,muy poco el pago con la referencia que gana diariamente uno
6,En la mutual me dicen que tengo el pago aprobado para cobrar en BancoEstado y en banco estado me dicen que no hay ningún pago a mi nombre hasta ahora solo he cobrado una de 3 licencias,en la mutual me dicen que tengo el pago aprobado para cobrar en bancoestado y en banco estado me dicen que no hay ningún pago a mi nombre hasta ahora solo he cobrado una de 3 licencias
7,Sigo esperando los pagos ya me dijeron que x fecha pagarian pero tuve yo que preguntar porque jamás dan información y de paso me contabilizaron mal el primer pago en fin no me gusto,sigo esperando los pagos ya me dijeron que x fecha pagarian pero tuve yo que preguntar porque jamás dan información y de paso me contabilizaron mal el primer pago en fin no me gusto
8,consulte en mutual x unos dias de reposo y quedaron de pagar el 1 de abril y no he tenido respuesta son 21 días de reposo y solo he recibido pago de 17 días,consulte en mutual x unos dias de reposo y quedaron de pagar el 1 de abril y no he tenido respuesta son 21 días de reposo y solo he recibido pago de 17 días
9,Listo,listo


In [10]:
def tokenizar_simple(texto):
    """Tokeniza mediante separación por espacios."""

    return texto.split()

df_comentarios["tokens"] = df_comentarios["comentario_limpio"].apply(tokenizar_simple)

display(df_comentarios[["comentario_limpio","tokens"]].head(20))
    

,comentario_limpio,tokens
0,en qué me hicieron 2 trasferencias de pago no sale si es el pago de licencia médica no informaron el monto ni nada,"[en, qué, me, hicieron, 2, trasferencias, de, pago, no, sale, si, es, el, pago, de, licencia, médica, no, informaron, el, monto, ni, nada]"
1,debería el monto ser tu sueldo base,"[debería, el, monto, ser, tu, sueldo, base]"
2,recibí el dinero pero no me llegó ningún correo o mensaje del respectivo pago,"[recibí, el, dinero, pero, no, me, llegó, ningún, correo, o, mensaje, del, respectivo, pago]"
3,la incertidumbre del día de fecha de pago ya que la página no me permite el acceso al portal online,"[la, incertidumbre, del, día, de, fecha, de, pago, ya, que, la, página, no, me, permite, el, acceso, al, portal, online]"
4,regular,[regular]
5,muy poco el pago con la referencia que gana diariamente uno,"[muy, poco, el, pago, con, la, referencia, que, gana, diariamente, uno]"
6,en la mutual me dicen que tengo el pago aprobado para cobrar en bancoestado y en banco estado me dicen que no hay ningún pago a mi nombre hasta ahora solo he cobrado una de 3 licencias,"[en, la, mutual, me, dicen, que, tengo, el, pago, aprobado, para, cobrar, en, bancoestado, y, en, banco, estado, me, dicen, que, no, hay, ningún, pago, a, mi, nombre, hasta, ahora, solo, he, cobrado, una, de, 3, licencias]"
7,sigo esperando los pagos ya me dijeron que x fecha pagarian pero tuve yo que preguntar porque jamás dan información y de paso me contabilizaron mal el primer pago en fin no me gusto,"[sigo, esperando, los, pagos, ya, me, dijeron, que, x, fecha, pagarian, pero, tuve, yo, que, preguntar, porque, jamás, dan, información, y, de, paso, me, contabilizaron, mal, el, primer, pago, en, fin, no, me, gusto]"
8,consulte en mutual x unos dias de reposo y quedaron de pagar el 1 de abril y no he tenido respuesta son 21 días de reposo y solo he recibido pago de 17 días,"[consulte, en, mutual, x, unos, dias, de, reposo, y, quedaron, de, pagar, el, 1, de, abril, y, no, he, tenido, respuesta, son, 21, días, de, reposo, y, solo, he, recibido, pago, de, 17, días]"
9,listo,[listo]


## Etapa 4: Generación de variantes de preprocesamiento

Las siguientes transformaciones se aplican de forma independiente para evaluar el efecto de distintas decisiones de normalización.

### 4.1 Eliminación controlada de stopwords

Se eliminan palabras frecuentes poco informativas, preservando negaciones relevantes para el significado del comentario.

In [11]:
STOPWORDS_BASE = {
    "el", "la", "los", "las",
    "un", "una", "unos", "unas",
    "de", "del", "al",
    "en", "por", "para",
    "con", "a",
    "y", "o",
    "es", "son", "fue", "ser",
    "lo", "se", "me", "mi", "mis",
    "su", "sus",
    "que"
}

NEGACIONES_A_MANTENER = {
    "no", "sin", "nunca", "ni", "tampoco"
}

STOPWORDS_CONTROLADAS = STOPWORDS_BASE - NEGACIONES_A_MANTENER

def remover_stopwords_controladas(tokens):
    """Elimina stopwords y conserva negaciones."""

    return [
        token for token in tokens
        if token not in STOPWORDS_CONTROLADAS
    ]

df_comentarios["tokens_sin_stopwords"] = df_comentarios["tokens"].apply(remover_stopwords_controladas)

display(df_comentarios[["tokens", "tokens_sin_stopwords"]].head(10))

,tokens,tokens_sin_stopwords
0,"[en, qué, me, hicieron, 2, trasferencias, de, pago, no, sale, si, es, el, pago, de, licencia, médica, no, informaron, el, monto, ni, nada]","[qué, hicieron, 2, trasferencias, pago, no, sale, si, pago, licencia, médica, no, informaron, monto, ni, nada]"
1,"[debería, el, monto, ser, tu, sueldo, base]","[debería, monto, tu, sueldo, base]"
2,"[recibí, el, dinero, pero, no, me, llegó, ningún, correo, o, mensaje, del, respectivo, pago]","[recibí, dinero, pero, no, llegó, ningún, correo, mensaje, respectivo, pago]"
3,"[la, incertidumbre, del, día, de, fecha, de, pago, ya, que, la, página, no, me, permite, el, acceso, al, portal, online]","[incertidumbre, día, fecha, pago, ya, página, no, permite, acceso, portal, online]"
4,[regular],[regular]
5,"[muy, poco, el, pago, con, la, referencia, que, gana, diariamente, uno]","[muy, poco, pago, referencia, gana, diariamente, uno]"
6,"[en, la, mutual, me, dicen, que, tengo, el, pago, aprobado, para, cobrar, en, bancoestado, y, en, banco, estado, me, dicen, que, no, hay, ningún, pago, a, mi, nombre, hasta, ahora, solo, he, cobrado, una, de, 3, licencias]","[mutual, dicen, tengo, pago, aprobado, cobrar, bancoestado, banco, estado, dicen, no, hay, ningún, pago, nombre, hasta, ahora, solo, he, cobrado, 3, licencias]"
7,"[sigo, esperando, los, pagos, ya, me, dijeron, que, x, fecha, pagarian, pero, tuve, yo, que, preguntar, porque, jamás, dan, información, y, de, paso, me, contabilizaron, mal, el, primer, pago, en, fin, no, me, gusto]","[sigo, esperando, pagos, ya, dijeron, x, fecha, pagarian, pero, tuve, yo, preguntar, porque, jamás, dan, información, paso, contabilizaron, mal, primer, pago, fin, no, gusto]"
8,"[consulte, en, mutual, x, unos, dias, de, reposo, y, quedaron, de, pagar, el, 1, de, abril, y, no, he, tenido, respuesta, son, 21, días, de, reposo, y, solo, he, recibido, pago, de, 17, días]","[consulte, mutual, x, dias, reposo, quedaron, pagar, 1, abril, no, he, tenido, respuesta, 21, días, reposo, solo, he, recibido, pago, 17, días]"
9,[listo],[listo]


### 4.2 Stemming

Se generan raíces aproximadas de las palabras mediante `SnowballStemmer` en español. Esta variante se evaluará de forma independiente durante el modelamiento.

In [12]:
def aplicar_stemming(tokens):
    """Aplica stemming en español."""
    
    return [stemmer_es.stem(token) for token in tokens]

if stemmer_es is not None:
    df_comentarios["tokens_stem"] = df_comentarios["tokens_sin_stopwords"].apply(aplicar_stemming)
    display(df_comentarios[["tokens_sin_stopwords", "tokens_stem"]].head(10))
else:
    print("Stemming no disponible. Instala nltk si quieres usar este bloque.")

,tokens_sin_stopwords,tokens_stem
0,"[qué, hicieron, 2, trasferencias, pago, no, sale, si, pago, licencia, médica, no, informaron, monto, ni, nada]","[que, hic, 2, trasferent, pag, no, sal, si, pag, licenci, medic, no, inform, mont, ni, nad]"
1,"[debería, monto, tu, sueldo, base]","[deb, mont, tu, sueld, bas]"
2,"[recibí, dinero, pero, no, llegó, ningún, correo, mensaje, respectivo, pago]","[recib, diner, per, no, lleg, ningun, corre, mensaj, respect, pag]"
3,"[incertidumbre, día, fecha, pago, ya, página, no, permite, acceso, portal, online]","[incertidumbr, dia, fech, pag, ya, pagin, no, permit, acces, portal, onlin]"
4,[regular],[regul]
5,"[muy, poco, pago, referencia, gana, diariamente, uno]","[muy, poc, pag, referent, gan, diari, uno]"
6,"[mutual, dicen, tengo, pago, aprobado, cobrar, bancoestado, banco, estado, dicen, no, hay, ningún, pago, nombre, hasta, ahora, solo, he, cobrado, 3, licencias]","[mutual, dic, teng, pag, aprob, cobr, bancoest, banc, estad, dic, no, hay, ningun, pag, nombr, hast, ahor, sol, he, cobr, 3, licenci]"
7,"[sigo, esperando, pagos, ya, dijeron, x, fecha, pagarian, pero, tuve, yo, preguntar, porque, jamás, dan, información, paso, contabilizaron, mal, primer, pago, fin, no, gusto]","[sig, esper, pag, ya, dijeron, x, fech, pagari, per, tuv, yo, pregunt, porqu, jamas, dan, inform, pas, contabiliz, mal, prim, pag, fin, no, gust]"
8,"[consulte, mutual, x, dias, reposo, quedaron, pagar, 1, abril, no, he, tenido, respuesta, 21, días, reposo, solo, he, recibido, pago, 17, días]","[consult, mutual, x, dias, repos, qued, pag, 1, abril, no, he, ten, respuest, 21, dias, repos, sol, he, recib, pag, 17, dias]"
9,[listo],[list]


### 4.3 Lematización

Se obtienen los lemas de las palabras mediante spaCy, cuando el modelo de idioma español se encuentra disponible.

In [13]:
def lematizar_texto(texto):
    """Obtiene lemas con spaCy."""

    if nlp is None:
        raise ImportError(
            "spaCy o el modelo es_core_news_sm no está disponible. "
            "Instala con: python -m spacy download es_core_news_sm"
        )

    doc = nlp(texto)

    lemas = []

    for token in doc:
        lema = token.lemma_.strip()

        if lema:
            lemas.append(lema)

    return lemas

if nlp is not None:
    df_comentarios["tokens_lemma"] = df_comentarios["comentario_limpio"].apply(lematizar_texto)
    display(df_comentarios[["comentario_limpio", "tokens_lemma"]].head(10))
else:
    print("Lematización no disponible. Instala spaCy y es_core_news_sm si quieres usar este bloque.")

,comentario_limpio,tokens_lemma
0,en qué me hicieron 2 trasferencias de pago no sale si es el pago de licencia médica no informaron el monto ni nada,"[en, qué, yo, hacer, 2, trasferencia, de, pago, no, salir, si, ser, el, pago, de, licencia, médico, no, informar, el, monto, ni, nada]"
1,debería el monto ser tu sueldo base,"[deber, el, monto, ser, tu, sueldo, base]"
2,recibí el dinero pero no me llegó ningún correo o mensaje del respectivo pago,"[recibir, el, dinero, pero, no, yo, llegar, ninguno, correo, o, mensaje, del, respectivo, pago]"
3,la incertidumbre del día de fecha de pago ya que la página no me permite el acceso al portal online,"[el, incertidumbre, del, día, de, fecha, de, pago, ya, que, el, página, no, yo, permitir, el, acceso, al, portal, online]"
4,regular,[regular]
5,muy poco el pago con la referencia que gana diariamente uno,"[mucho, poco, el, pago, con, el, referencia, que, ganar, diariamente, uno]"
6,en la mutual me dicen que tengo el pago aprobado para cobrar en bancoestado y en banco estado me dicen que no hay ningún pago a mi nombre hasta ahora solo he cobrado una de 3 licencias,"[en, el, mutual, yo, decir, que, tener, el, pago, aprobado, para, cobrar, en, bancoestado, y, en, banco, estado, yo, decir, que, no, haber, ninguno, pago, a, mi, nombre, hasta, ahora, solo, haber, cobrar, uno, de, 3, licencia]"
7,sigo esperando los pagos ya me dijeron que x fecha pagarian pero tuve yo que preguntar porque jamás dan información y de paso me contabilizaron mal el primer pago en fin no me gusto,"[seguir, esperar, el, pago, ya, yo, decir, que, x, fecha, pagariar, pero, tener, yo, que, preguntar, porque, jamás, dar, información, y, de, paso, yo, contabilizar, mal, el, primero, pago, en, fin, no, yo, gustar]"
8,consulte en mutual x unos dias de reposo y quedaron de pagar el 1 de abril y no he tenido respuesta son 21 días de reposo y solo he recibido pago de 17 días,"[consultar, en, mutual, x, uno, dia, de, reposo, y, quedar, de, pagar, el, 1, de, abril, y, no, haber, tener, respuesta, ser, 21, día, de, reposo, y, solo, haber, recibir, pago, de, 17, día]"
9,listo,[listo]


### 4.4 Normalización sin tildes

Se crea una variante sin tildes que preserva la letra `ñ`, con el objetivo de reducir variantes ortográficas sin alterar términos cuyo significado cambia al eliminarla.

In [14]:
import unicodedata

def quitar_tildes_preservando_enie(texto):
    """Elimina tildes y preserva ñ."""

    if not isinstance(texto, str):
        return ""

    texto = texto.replace("ñ", "__enie__")
    texto = texto.replace("Ñ", "__ENIE__")

    texto = unicodedata.normalize("NFD", texto)

    texto = "".join(
        caracter for caracter in texto
        if unicodedata.category(caracter) != "Mn"
    )

    texto = texto.replace("__enie__", "ñ")
    texto = texto.replace("__ENIE__", "Ñ")

    texto = unicodedata.normalize("NFC", texto)

    return texto

df_comentarios["comentario_limpio_sin_tildes"] = (
    df_comentarios["comentario_limpio"]
    .apply(quitar_tildes_preservando_enie)
)

df_comentarios["tokens_sin_tildes"] = df_comentarios["comentario_limpio_sin_tildes"].apply(
    lambda texto: texto.split() if isinstance(texto, str) else []
)

display(
    df_comentarios[
        ["comentario_limpio", "comentario_limpio_sin_tildes"]
    ].head(20)
)

,comentario_limpio,comentario_limpio_sin_tildes
0,en qué me hicieron 2 trasferencias de pago no sale si es el pago de licencia médica no informaron el monto ni nada,en que me hicieron 2 trasferencias de pago no sale si es el pago de licencia medica no informaron el monto ni nada
1,debería el monto ser tu sueldo base,deberia el monto ser tu sueldo base
2,recibí el dinero pero no me llegó ningún correo o mensaje del respectivo pago,recibi el dinero pero no me llego ningun correo o mensaje del respectivo pago
3,la incertidumbre del día de fecha de pago ya que la página no me permite el acceso al portal online,la incertidumbre del dia de fecha de pago ya que la pagina no me permite el acceso al portal online
4,regular,regular
5,muy poco el pago con la referencia que gana diariamente uno,muy poco el pago con la referencia que gana diariamente uno
6,en la mutual me dicen que tengo el pago aprobado para cobrar en bancoestado y en banco estado me dicen que no hay ningún pago a mi nombre hasta ahora solo he cobrado una de 3 licencias,en la mutual me dicen que tengo el pago aprobado para cobrar en bancoestado y en banco estado me dicen que no hay ningun pago a mi nombre hasta ahora solo he cobrado una de 3 licencias
7,sigo esperando los pagos ya me dijeron que x fecha pagarian pero tuve yo que preguntar porque jamás dan información y de paso me contabilizaron mal el primer pago en fin no me gusto,sigo esperando los pagos ya me dijeron que x fecha pagarian pero tuve yo que preguntar porque jamas dan informacion y de paso me contabilizaron mal el primer pago en fin no me gusto
8,consulte en mutual x unos dias de reposo y quedaron de pagar el 1 de abril y no he tenido respuesta son 21 días de reposo y solo he recibido pago de 17 días,consulte en mutual x unos dias de reposo y quedaron de pagar el 1 de abril y no he tenido respuesta son 21 dias de reposo y solo he recibido pago de 17 dias
9,listo,listo


### 4.5 Consolidación de versiones de texto

Se construyen las cinco columnas que se utilizarán en los experimentos: texto limpio, sin tildes, sin stopwords, con stemming y con lematización.

In [15]:
df_comentarios["texto_v1_limpio"] = df_comentarios["comentario_limpio"]

df_comentarios["texto_v2_sin_tildes"] = df_comentarios["comentario_limpio_sin_tildes"]

df_comentarios["texto_v3_sin_stopwords"] = df_comentarios["tokens_sin_stopwords"].apply(
    lambda tokens: " ".join(tokens)
)

if "tokens_stem" in df_comentarios.columns:
    df_comentarios["texto_v4_stemming"] = df_comentarios["tokens_stem"].apply(
        lambda tokens: " ".join(tokens)
    )

if "tokens_lemma" in df_comentarios.columns:
    df_comentarios["texto_v5_lemma"] = df_comentarios["tokens_lemma"].apply(
        lambda tokens: " ".join(tokens)
    )

columnas_versiones = [
    "comentario_original",
    "texto_v1_limpio",
    "texto_v2_sin_tildes",
    "texto_v3_sin_stopwords"
]

if "texto_v4_stemming" in df_comentarios.columns:
    columnas_versiones.append("texto_v4_stemming")

if "texto_v5_lemma" in df_comentarios.columns:
    columnas_versiones.append("texto_v5_lemma")

display(df_comentarios.head(10))

,ResponseId,pregunta_origen,comentario_original,comentario_limpio,tokens,tokens_sin_stopwords,tokens_stem,tokens_lemma,comentario_limpio_sin_tildes,tokens_sin_tildes,texto_v1_limpio,texto_v2_sin_tildes,texto_v3_sin_stopwords,texto_v4_stemming,texto_v5_lemma
0,29,Q2_10_TEXT,En qué me hicieron 2 trasferencias de pago no sale si es el pago de licencia médica no informaron el monto ni nada,en qué me hicieron 2 trasferencias de pago no sale si es el pago de licencia médica no informaron el monto ni nada,"[en, qué, me, hicieron, 2, trasferencias, de, pago, no, sale, si, es, el, pago, de, licencia, médica, no, informaron, el, monto, ni, nada]","[qué, hicieron, 2, trasferencias, pago, no, sale, si, pago, licencia, médica, no, informaron, monto, ni, nada]","[que, hic, 2, trasferent, pag, no, sal, si, pag, licenci, medic, no, inform, mont, ni, nad]","[en, qué, yo, hacer, 2, trasferencia, de, pago, no, salir, si, ser, el, pago, de, licencia, médico, no, informar, el, monto, ni, nada]",en que me hicieron 2 trasferencias de pago no sale si es el pago de licencia medica no informaron el monto ni nada,"[en, que, me, hicieron, 2, trasferencias, de, pago, no, sale, si, es, el, pago, de, licencia, medica, no, informaron, el, monto, ni, nada]",en qué me hicieron 2 trasferencias de pago no sale si es el pago de licencia médica no informaron el monto ni nada,en que me hicieron 2 trasferencias de pago no sale si es el pago de licencia medica no informaron el monto ni nada,qué hicieron 2 trasferencias pago no sale si pago licencia médica no informaron monto ni nada,que hic 2 trasferent pag no sal si pag licenci medic no inform mont ni nad,en qué yo hacer 2 trasferencia de pago no salir si ser el pago de licencia médico no informar el monto ni nada
1,35,Q2_10_TEXT,Debería el monto ser tu sueldo base,debería el monto ser tu sueldo base,"[debería, el, monto, ser, tu, sueldo, base]","[debería, monto, tu, sueldo, base]","[deb, mont, tu, sueld, bas]","[deber, el, monto, ser, tu, sueldo, base]",deberia el monto ser tu sueldo base,"[deberia, el, monto, ser, tu, sueldo, base]",debería el monto ser tu sueldo base,deberia el monto ser tu sueldo base,debería monto tu sueldo base,deb mont tu sueld bas,deber el monto ser tu sueldo base
2,39,Q2_10_TEXT,"Recibí el dinero, pero no me llegó ningún correo o mensaje del respectivo pago",recibí el dinero pero no me llegó ningún correo o mensaje del respectivo pago,"[recibí, el, dinero, pero, no, me, llegó, ningún, correo, o, mensaje, del, respectivo, pago]","[recibí, dinero, pero, no, llegó, ningún, correo, mensaje, respectivo, pago]","[recib, diner, per, no, lleg, ningun, corre, mensaj, respect, pag]","[recibir, el, dinero, pero, no, yo, llegar, ninguno, correo, o, mensaje, del, respectivo, pago]",recibi el dinero pero no me llego ningun correo o mensaje del respectivo pago,"[recibi, el, dinero, pero, no, me, llego, ningun, correo, o, mensaje, del, respectivo, pago]",recibí el dinero pero no me llegó ningún correo o mensaje del respectivo pago,recibi el dinero pero no me llego ningun correo o mensaje del respectivo pago,recibí dinero pero no llegó ningún correo mensaje respectivo pago,recib diner per no lleg ningun corre mensaj respect pag,recibir el dinero pero no yo llegar ninguno correo o mensaje del respectivo pago
3,73,Q2_10_TEXT,La incertidumbre del día de fecha de pago ya que la página no me permite el acceso al portal online,la incertidumbre del día de fecha de pago ya que la página no me permite el acceso al portal online,"[la, incertidumbre, del, día, de, fecha, de, pago, ya, que, la, página, no, me, permite, el, acceso, al, portal, online]","[incertidumbre, día, fecha, pago, ya, página, no, permite, acceso, portal, online]","[incertidumbr, dia, fech, pag, ya, pagin, no, permit, acces, portal, onlin]","[el, incertidumbre, del, día, de, fecha, de, pago, ya, que, el, página, no, yo, permitir, el, acceso, al, portal, online]",la incertidumbre del dia de fecha de pago ya que la pagina no me permite el acceso al p

## Etapa 5: Extracción de características

Las versiones textuales se preparan para su representación numérica. Se mantiene una base común con `idx_feature_matrix` para asegurar la alineación entre comentarios, matrices de características y etiquetas.

### 5.1 Base común para las representaciones

Se crea `df_features_base` y se seleccionan inicialmente las columnas utilizadas por TF-IDF y Word2Vec.

In [16]:
df_features_base = df_comentarios.copy()

VERSION_TEXTO_NGRAMS = "comentario_limpio"

if VERSION_TEXTO_NGRAMS not in df_features_base.columns:
    raise ValueError(f"No existe la columna seleccionada para n-grams: {VERSION_TEXTO_NGRAMS}")

df_features_base["texto_para_ngrams"] = df_features_base[VERSION_TEXTO_NGRAMS].astype(str)

VERSION_TOKENS_W2V = "tokens"

if VERSION_TOKENS_W2V not in df_features_base.columns:
    raise ValueError(f"No existe la columna seleccionada para Word2Vec: {VERSION_TOKENS_W2V}")

df_features_base["tokens_para_word2vec"] = df_features_base[VERSION_TOKENS_W2V]

def asegurar_lista_tokens(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    return str(x).split()

df_features_base["tokens_para_word2vec"] = df_features_base["tokens_para_word2vec"].apply(asegurar_lista_tokens)

df_features_base["idx_feature_matrix"] = range(len(df_features_base))

print("Base para Feature Extraction creada correctamente.")
print("Filas:", len(df_features_base))
print("Texto usado para n-grams:", VERSION_TEXTO_NGRAMS)
print("Tokens usados para Word2Vec:", VERSION_TOKENS_W2V)

display(
    df_features_base[
        [
            "idx_feature_matrix",
            "pregunta_origen",
            "comentario_original",
            "texto_para_ngrams",
            "tokens_para_word2vec",
        ]
    ].tail()
)

Base para Feature Extraction creada correctamente.
Filas: 6119
Texto usado para n-grams: comentario_limpio
Tokens usados para Word2Vec: tokens


,idx_feature_matrix,pregunta_origen,comentario_original,texto_para_ngrams,tokens_para_word2vec
6114,6114,Q5,Puntualidad,puntualidad,[puntualidad]
6115,6115,Q5,La puntualidad en el pago,la puntualidad en el pago,"[la, puntualidad, en, el, pago]"
6116,6116,Q5,Buena remuneración por dia de licencia,buena remuneración por dia de licencia,"[buena, remuneración, por, dia, de, licencia]"
6117,6117,Q5,No tener ningún inconveniente con mi pago,no tener ningún inconveniente con mi pago,"[no, tener, ningún, inconveniente, con, mi, pago]"
6118,6118,Q5,Transparencia,transparencia,[transparencia]


### 5.2 Representación TF-IDF con n-gramas

Se construye una matriz dispersa con unigramas y bigramas. Esta representación asigna peso a los términos según su importancia relativa en el corpus.

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy import sparse
import joblib

vectorizer_tfidf_ngrams = TfidfVectorizer(
    lowercase=False,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    dtype=np.float32
)

X_tfidf_ngrams = vectorizer_tfidf_ngrams.fit_transform(
    df_features_base["texto_para_ngrams"]
)

feature_names_tfidf_ngrams = vectorizer_tfidf_ngrams.get_feature_names_out()

print("Matriz TF-IDF + N-grams creada correctamente.")
print("Filas/comentarios:", X_tfidf_ngrams.shape[0])
print("Columnas/features:", X_tfidf_ngrams.shape[1])
print("Tipo de matriz:", type(X_tfidf_ngrams))

print("\nPrimeras 50 features:")
print(feature_names_tfidf_ngrams[:50])

Matriz TF-IDF + N-grams creada correctamente.
Filas/comentarios: 6119
Columnas/features: 7110
Tipo de matriz: <class 'scipy.sparse._csr.csr_matrix'>

Primeras 50 features:
['00' '00 por' '03' '05' '06' '07' '08' '08 22' '10' '10 10' '10 de'
 '10 dias' '10 días' '100' '100 del' '11' '11 de' '11 dias' '11 días' '12'
 '12 dias' '12 días' '12 me' '13' '13 de' '13 dias' '14' '14 dias'
 '14 días' '15' '15 15' '15 30' '15 de' '15 dia' '15 dias' '15 día'
 '15 días' '15 los' '15 numero' '16' '16 744' '17' '18' '18 de' '18 días'
 '18 mil' '19' '20' '20 dias' '20 días']


### 5.3 DataFrame de referencia para TF-IDF

Se crea un DataFrame liviano que permite relacionar cada fila de la matriz TF-IDF con su comentario y la versión de texto empleada.

In [18]:
df_ngram_version = df_features_base[
    [
        "idx_feature_matrix",
        "ResponseId",
        "pregunta_origen",
        "comentario_original",
        "texto_para_ngrams"
    ]
].copy()

df_ngram_version["feature_extraction_method"] = "tfidf_ngrams"
df_ngram_version["feature_extraction_detail"] = "tfidf_ngram_1_2_min_df_2_max_df_095"
df_ngram_version["texto_base_usado"] = VERSION_TEXTO_NGRAMS

display(df_ngram_version.head(10))

print("df_ngram_version creado.")
print("Recuerda: las features numéricas están en X_tfidf_ngrams.")

,idx_feature_matrix,ResponseId,pregunta_origen,comentario_original,texto_para_ngrams,feature_extraction_method,feature_extraction_detail,texto_base_usado
0,0,29,Q2_10_TEXT,En qué me hicieron 2 trasferencias de pago no sale si es el pago de licencia médica no informaron el monto ni nada,en qué me hicieron 2 trasferencias de pago no sale si es el pago de licencia médica no informaron el monto ni nada,tfidf_ngrams,tfidf_ngram_1_2_min_df_2_max_df_095,comentario_limpio
1,1,35,Q2_10_TEXT,Debería el monto ser tu sueldo base,debería el monto ser tu sueldo base,tfidf_ngrams,tfidf_ngram_1_2_min_df_2_max_df_095,comentario_limpio
2,2,39,Q2_10_TEXT,"Recibí el dinero, pero no me llegó ningún correo o mensaje del respectivo pago",recibí el dinero pero no me llegó ningún correo o mensaje del respectivo pago,tfidf_ngrams,tfidf_ngram_1_2_min_df_2_max_df_095,comentario_limpio
3,3,73,Q2_10_TEXT,La incertidumbre del día de fecha de pago ya que la página no me permite el acceso al portal online,la incertidumbre del día de fecha de pago ya que la página no me permite el acceso al portal online,tfidf_ngrams,tfidf_ngram_1_2_min_df_2_max_df_095,comentario_limpio
4,4,83,Q2_10_TEXT,Regular,regular,tfidf_ngrams,tfidf_ngram_1_2_min_df_2_max_df_095,comentario_limpio
5,5,91,Q2_10_TEXT,Muy poco el pago con la referencia que gana diariamente uno,muy poco el pago con la referencia que gana diariamente uno,tfidf_ngrams,tfidf_ngram_1_2_min_df_2_max_df_095,comentario_limpio
6,6,132,Q2_10_TEXT,En la mutual me dicen que tengo el pago aprobado para cobrar en BancoEstado y en banco estado me dicen que no hay ningún pago a mi nombre hasta ahora solo he cobrado una de 3 licencias,en la mutual me dicen que tengo el pago aprobado para cobrar en bancoestado y en banco estado me dicen que no hay ningún pago a mi nombre hasta ahora solo he cobrado una de 3 licencias,tfidf_ngrams,tfidf_ngram_1_2_min_df_2_max_df_095,comentario_limpio
7,7,134,Q2_10_TEXT,Sigo esperando los pagos ya me dijeron que x fecha pagarian pero tuve yo que preguntar porque jamás dan información y de paso me contabilizaron mal el primer pago en fin no me gusto,sigo esperando los pagos ya me dijeron que x fecha pagarian pero tuve yo que preguntar porque jamás dan información y de paso me contabilizaron mal el primer pago en fin no me gusto,tfidf_ngrams,tfidf_ngram_1_2_min_df_2_max_df_095,comentario_limpio
8,8,145,Q2_10_TEXT,consulte en mutual x unos dias de reposo y quedaron de pagar el 1 de abril y no he tenido respuesta son 21 días de reposo y solo he recibido pago de 17 días,consulte en mutual x unos dias de reposo y quedaron de pagar el 1 de abril y no he tenido respuesta son 21 días de reposo y solo he recibido pago de 17 días,tfidf_ngrams,tfidf_ngram_1_2_min_df_2_max_df_095,comentario_limpio
9,9,157,Q2_10_TEXT,Listo,listo,tfidf_ngrams,tfidf_ngram_1_2_min_df_2_max_df_095,comentario_limpio


df_ngram_version creado.
Recuerda: las features numéricas están en X_tfidf_ngrams.


### 5.4 Inspección de features TF-IDF

Se define una función de apoyo para revisar los términos o n-gramas con mayor peso en un comentario específico.

In [19]:
def ver_top_features_tfidf(fila_idx, top_n=15):
    """Devuelve las features TF-IDF de mayor peso."""

    fila = X_tfidf_ngrams[fila_idx]

    indices = fila.indices
    valores = fila.data

    df_temp = pd.DataFrame({
        "feature": feature_names_tfidf_ngrams[indices],
        "tfidf": valores
    })

    df_temp = df_temp.sort_values("tfidf", ascending=False).head(top_n)

    return df_temp

fila_ejemplo = 0

print("Comentario original:")
print(df_ngram_version.loc[fila_ejemplo, "comentario_original"])

print("\nTexto usado para TF-IDF:")
print(df_ngram_version.loc[fila_ejemplo, "texto_para_ngrams"])

print("\nTop features TF-IDF:")
display(ver_top_features_tfidf(fila_ejemplo, top_n=15))

Comentario original:
En qué me hicieron 2 trasferencias de pago no sale si es el pago de licencia médica no informaron el monto ni nada

Texto usado para TF-IDF:
en qué me hicieron 2 trasferencias de pago no sale si es el pago de licencia médica no informaron el monto ni nada

Top features TF-IDF:


,feature,tfidf
31,ni nada,0.263593
17,en qué,0.254796
22,no sale,0.254796
18,qué me,0.247973
29,médica no,0.242398
7,sale,0.230001
13,informaron,0.230001
19,me hicieron,0.223865
23,si es,0.218757
3,hicieron,0.202671


### 5.5 Preparación de Word2Vec

Se carga la dependencia necesaria para entrenar representaciones densas de palabras con Word2Vec.

In [21]:
try:
    from gensim.models import Word2Vec
except ImportError as e:
    raise ImportError(
        "No está instalada la librería gensim. "
        "Instálala con: %pip install gensim"
    ) from e

### 5.6 Entrenamiento de Word2Vec

Se entrena un modelo Skip-gram sobre los tokens de los comentarios, usando una dimensión vectorial fija y parámetros reproducibles.

In [22]:
corpus_tokens_w2v = df_features_base["tokens_para_word2vec"].tolist()

corpus_tokens_w2v = [tokens for tokens in corpus_tokens_w2v if len(tokens) > 0]

VECTOR_SIZE_W2V = 100

modelo_w2v = Word2Vec(
    sentences=corpus_tokens_w2v,
    vector_size=VECTOR_SIZE_W2V,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    seed=42
)

print("Modelo Word2Vec entrenado correctamente.")
print("Tamaño del vocabulario Word2Vec:", len(modelo_w2v.wv.index_to_key))
print("Dimensión de vectores:", modelo_w2v.vector_size)

print("\nPrimeras 30 palabras del vocabulario:")
print(modelo_w2v.wv.index_to_key[:30])

Modelo Word2Vec entrenado correctamente.
Tamaño del vocabulario Word2Vec: 2193
Dimensión de vectores: 100

Primeras 30 palabras del vocabulario:
['la', 'de', 'que', 'el', 'y', 'pago', 'no', 'en', 'me', 'a', 'licencia', 'los', 'mi', 'se', 'puntualidad', 'por', 'muy', 'es', 'lo', 'con', 'atención', 'las', 'del', 'para', 'un', 'todo', 'tiempo', 'fue', 'fecha', 'al']


### 5.7 Inspección de similitudes semánticas

Se revisan palabras similares en el espacio de embeddings como control exploratorio del modelo Word2Vec.

In [23]:
palabras_prueba = [
    "pago",
    "licencia",
    "médica",
    "correo",
    "monto",
    "portal",
    "informaron"
]

for palabra in palabras_prueba:
    if palabra in modelo_w2v.wv:
        print(f"\nPalabras similares a '{palabra}':")
        display(pd.DataFrame(modelo_w2v.wv.most_similar(palabra, topn=10), columns=["palabra", "similitud"]))
    else:
        print(f"\nLa palabra '{palabra}' no está en el vocabulario Word2Vec.")


Palabras similares a 'pago':


,palabra,similitud
0,tiempo,0.953791
1,oportuno,0.905635
2,indicada,0.904609
3,cumplimiento,0.900301
4,espera,0.885833
5,demora,0.879894
6,reposo,0.878507
7,de,0.864899
8,monto,0.864607
9,antes,0.856421



Palabras similares a 'licencia':


,palabra,similitud
0,reposo,0.946413
1,antes,0.925394
2,demora,0.921607
3,trayecto,0.921267
4,total,0.918684
5,tu,0.918639
6,trabajo,0.908802
7,monto,0.904360
8,espera,0.903829
9,dia,0.898492



Palabras similares a 'médica':


...

,palabra,similitud
0,cálculo,0.974875
1,total,0.972603
2,dia,0.971413
3,detalle,0.970640
4,valor,0.969603
5,bajo,0.967826
6,antes,0.966256
7,pagó,0.963261
8,día,0.961702
9,correspondiente,0.960984



Palabras similares a 'portal':


,palabra,similitud
0,creen,0.996791
1,hacerme,0.996474
2,mena,0.996339
3,pedir,0.996288
4,examen,0.996182
5,izquierda,0.996143
6,puedan,0.996135
7,zona,0.996123
8,estamos,0.996097
9,juicio,0.995990



Palabras similares a 'informaron':


,palabra,similitud
0,mayo,0.997890
1,semana,0.997678
2,estando,0.997578
3,periodo,0.997414
4,estaría,0.997346
5,haya,0.997304
6,vía,0.997287
7,calculan,0.997284
8,realizo,0.997283
9,recién,0.997242


### 5.8 Representación de comentarios mediante Word2Vec promedio

Cada comentario se convierte en un vector denso a partir del promedio de los embeddings de los tokens presentes en su vocabulario.

In [24]:
def vector_promedio_word2vec(tokens, modelo, vector_size):
    """Promedia embeddings de los tokens del comentario."""

    if not isinstance(tokens, list):
        tokens = asegurar_lista_tokens(tokens)

    vectores = []

    for token in tokens:
        if token in modelo.wv:
            vectores.append(modelo.wv[token])

    if len(vectores) == 0:
        return np.zeros(vector_size, dtype=np.float32)

    return np.mean(vectores, axis=0).astype(np.float32)

X_word2vec = np.vstack(
    df_features_base["tokens_para_word2vec"].apply(
        lambda tokens: vector_promedio_word2vec(
            tokens=tokens,
            modelo=modelo_w2v,
            vector_size=modelo_w2v.vector_size
        )
    )
)

print("Matriz Word2Vec promedio creada correctamente.")
print("Filas/comentarios:", X_word2vec.shape[0])
print("Columnas/dimensiones:", X_word2vec.shape[1])
print("Tipo:", type(X_word2vec))

Matriz Word2Vec promedio creada correctamente.
Filas/comentarios: 6119
Columnas/dimensiones: 100
Tipo: <class 'numpy.ndarray'>


### 5.9 DataFrame de referencia para Word2Vec

Se construye un DataFrame que integra la información del comentario con las dimensiones numéricas generadas por Word2Vec.

In [25]:
columnas_w2v = [f"w2v_{i:03d}" for i in range(modelo_w2v.vector_size)]

df_word2vec_features = pd.DataFrame(
    X_word2vec,
    columns=columnas_w2v
)

df_word2vec_version = pd.concat(
    [
        df_features_base[
            [
                "idx_feature_matrix",
                "ResponseId",
                "pregunta_origen",
                "comentario_original",
                "comentario_limpio"
            ]
        ].reset_index(drop=True),
        df_word2vec_features.reset_index(drop=True)
    ],
    axis=1
)

df_word2vec_version["feature_extraction_method"] = "word2vec"
df_word2vec_version["feature_extraction_detail"] = f"word2vec_skipgram_avg_{modelo_w2v.vector_size}d"
df_word2vec_version["tokens_base_usados"] = VERSION_TOKENS_W2V

display(df_word2vec_version.head())

print("df_word2vec_version creado.")
print("Las columnas w2v_000 ... w2v_099 son las features numéricas.")

,idx_feature_matrix,ResponseId,pregunta_origen,comentario_original,comentario_limpio,w2v_000,w2v_001,w2v_002,w2v_003,w2v_004,w2v_005,w2v_006,w2v_007,w2v_008,w2v_009,w2v_010,w2v_011,w2v_012,w2v_013,w2v_014,w2v_015,w2v_016,w2v_017,w2v_018,w2v_019,w2v_020,w2v_021,w2v_022,w2v_023,w2v_024,w2v_025,w2v_026,w2v_027,w2v_028,w2v_029,w2v_030,w2v_031,w2v_032,w2v_033,w2v_034,w2v_035,w2v_036,w2v_037,w2v_038,w2v_039,w2v_040,w2v_041,w2v_042,w2v_043,w2v_044,w2v_045,w2v_046,w2v_047,w2v_048,w2v_049,w2v_050,w2v_051,w2v_052,w2v_053,w2v_054,w2v_055,w2v_056,w2v_057,w2v_058,w2v_059,w2v_060,w2v_061,w2v_062,w2v_063,w2v_064,w2v_065,w2v_066,w2v_067,w2v_068,w2v_069,w2v_070,w2v_071,w2v_072,w2v_073,w2v_074,w2v_075,w2v_076,w2v_077,w2v_078,w2v_079,w2v_080,w2v_081,w2v_082,w2v_083,w2v_084,w2v_085,w2v_086,w2v_087,w2v_088,w2v_089,w2v_090,w2v_091,w2v_092,w2v_093,w2v_094,w2v_095,w2v_096,w2v_097,w2v_098,w2v_099,feature_extraction_method,feature_extraction_detail,tokens_base_usados
0,0,29,Q2_10_TEXT,En qué me hicieron 2 trasferencias de pago no sale si es el pago de licencia médica no informaron el monto ni nada,en qué me hicieron 2 trasferencias de pago no sale si es el pago de licencia médica no informaron el monto ni nada,-0.051628,0.060019,0.039720,0.231960,-0.024810,0.095711,0.017827,0.154164,-0.263475,-0.001466,-0.080078,0.222825,0.134878,0.179778,0.017928,-0.027108,-0.010749,-0.308994,-0.079764,-0.034188,-0.177470,-0.078241,0.053629,0.295667,0.386500,-0.191312,-0.144200,-0.052264,-0.227241,-0.121947,-0.145216,-0.194288,-0.085838,0.020329,-0.053232,0.013601,0.226415,-0.386770,-0.197690,0.169491,0.103149,0.125379,0.357651,-0.301878,0.092878,-0.015703,0.120376,0.088196,-0.063680,-0.118799,0.182235,-0.337835,0.005686,-0.001909,0.132111,0.098939,-0.036881,0.100872,-0.043275,0.311160,0.062367,0.044250,-0.209950,0.230703,0.091359,0.045397,0.040396,-0.162737,-0.197376,-0.197460,0.007129,-0.239066,0.039391,-0.112772,-0.037032,-0.184436,0.031661,0.026011,-0.107241,-0.012858,-0.082615,0.101893,-0.045042,0.051547,-0.247700,0.032174,-0.165562,-0.159988,0.059078,-0.053248,0.023521,0.153409,-0.048352,-0.084541,0.245495,-0.060722,-0.074605,-0.042159,-0.221901,-0.090821,word2vec,word2vec_skipgram_avg_100d,tokens
1,1,35,Q2_10_TEXT,Debería el monto ser tu sueldo base,debería el monto ser tu sueldo base,-0.068148,0.025630,0.036568,0.222441,-0.034136,0.103192,0.044946,0.172664,-0.216075,-0.026916,-0.052508,0.148394,0.109027,0.149810,0.015226,-0.043704,0.012574,-0.270914,-0.049175,-0.052230,-0.166435,-0.104125,0.030038,0.307571,0.386207,-0.150988,-0.148890,-0.049692,-0.201835,-0.173390,-0.154670,-0.183871,-0.055004,0.048097,-0.066340,0.015765,0.199580,-0.343306,-0.224200,0.156333,0.105230,0.146026,0.326669,-0.266748,0.137431,-0.013028,0.170596,0.140504,-0.095305,-0.122909,0.142241,-0.316552,0.028369,-0.041453,0.120654,0.134967,-0.087107,0.125880,-0.038179,0.319979,0.011797,0.023141,-0.209289,0.219997,0.080998,0.029715,0.004268,-0.134509,-0.232619,-0.152183,-0.001352,-0.297717,0.075311,-0.122619,-0.051628,-0.209713,0.072022,0.092296,-0.118234,-0.006822,-0.104038,0.118108,-0.035738,0.024642,-0.284450,-0.019218,-0.113834,-0.145941,0.055723,-0.047621,0.001159,0.174372,-0.070368,-0.139545,0.263631,-0.054187,-0.097713,-0.051347,-0.277970,-0.098694,word2vec,word2vec_skipgram_avg_100d,tokens
2,2,39,Q2_10_TEXT,"Recibí el dinero, pero no me llegó ningún correo o mensaje del respectivo pago",recibí el dinero pero no me llegó ningún correo o mensaje del respectivo pago,-0.061469,0.033530,0.018898,0.222298,-0.047888,0.095064,0.049250,0.139905,-0.191158,-0.021304,-0.083418,0.182552,0.112907,0.188486,0.027096,-0.039698,0.026492,-0.269300,-0.064686,-0.025977,-0.135361,-0.079130,0.003828,0.322321,0.336172,-0.165432,-0.146697,-0.004528,-0.205472,-0.123727,-0.138907,-0.178341,-0.063332,0.027579,-0.038178,0.041152,0.225423,-0.349076,-0.194103,0.154537,0.064939,0.148751,0.348563,-0.269633,0.079704,0.013442,0.121531,0.103075,-0.059797,-0.115530,0.131773,-0.363481,0.038271,-0.011846,0.105554,0.103537,-0.

df_word2vec_version creado.
Las columnas w2v_000 ... w2v_099 son las features numéricas.


### 5.10 Validación de alineación

Se verifica que TF-IDF, Word2Vec y sus DataFrames de referencia representen los mismos comentarios y mantengan el mismo orden de filas.

In [26]:
assert X_tfidf_ngrams.shape[0] == len(df_features_base), "TF-IDF no tiene la misma cantidad de filas que df_features_base"
assert X_word2vec.shape[0] == len(df_features_base), "Word2Vec no tiene la misma cantidad de filas que df_features_base"
assert len(df_ngram_version) == len(df_features_base), "df_ngram_version no está alineado"
assert len(df_word2vec_version) == len(df_features_base), "df_word2vec_version no está alineado"

print("Validación correcta: ambas representaciones tienen la misma cantidad de comentarios.")
print("Comentarios:", len(df_features_base))
print("TF-IDF shape:", X_tfidf_ngrams.shape)
print("Word2Vec shape:", X_word2vec.shape)

display(
    pd.DataFrame({
        "objeto": [
            "df_features_base",
            "X_tfidf_ngrams",
            "df_ngram_version",
            "X_word2vec",
            "df_word2vec_version"
        ],
        "filas": [
            len(df_features_base),
            X_tfidf_ngrams.shape[0],
            len(df_ngram_version),
            X_word2vec.shape[0],
            len(df_word2vec_version)
        ],
        "columnas": [
            df_features_base.shape[1],
            X_tfidf_ngrams.shape[1],
            df_ngram_version.shape[1],
            X_word2vec.shape[1],
            df_word2vec_version.shape[1]
        ]
    })
)

Validación correcta: ambas representaciones tienen la misma cantidad de comentarios.
Comentarios: 6119
TF-IDF shape: (6119, 7110)
Word2Vec shape: (6119, 100)


,objeto,filas,columnas
0,df_features_base,6119,18
1,X_tfidf_ngrams,6119,7110
2,df_ngram_version,6119,8
3,X_word2vec,6119,100
4,df_word2vec_version,6119,108


### 5.11 Guardado de artefactos de extracción de características

Se guardan las matrices, vectorizadores, modelos y DataFrames de referencia para reutilizar las representaciones sin recalcularlas.

In [27]:
RUTA_FEATURES = Path("features_texto")
RUTA_FEATURES.mkdir(exist_ok=True)

sparse.save_npz(RUTA_FEATURES / "X_tfidf_ngrams.npz", X_tfidf_ngrams)

joblib.dump(vectorizer_tfidf_ngrams, RUTA_FEATURES / "vectorizer_tfidf_ngrams.joblib")

df_ngram_version.to_csv(
    RUTA_FEATURES / "df_ngram_version.csv",
    index=False,
    encoding="utf-8-sig"
)

np.save(RUTA_FEATURES / "X_word2vec.npy", X_word2vec)

df_word2vec_version.to_csv(
    RUTA_FEATURES / "df_word2vec_version.csv",
    index=False,
    encoding="utf-8-sig"
)

modelo_w2v.save(str(RUTA_FEATURES / "modelo_word2vec_comentarios.model"))

print("Objetos de Feature Extraction guardados en:", RUTA_FEATURES.resolve())

Objetos de Feature Extraction guardados en: H:\Mi unidad\Proyecto ML Universidad\features_texto


## Etapa 6: Exportación para etiquetado manual

Se generan dos archivos Excel alineados con el universo de comentarios: uno asociado a TF-IDF con n-gramas y otro a Word2Vec. Ambos incluyen las versiones de texto necesarias para etiquetar manualmente las categorías.

In [28]:
from pathlib import Path

objetos_requeridos = [
    "df_features_base",
    "df_ngram_version",
    "df_word2vec_version"
]

faltantes = [
    objeto for objeto in objetos_requeridos
    if objeto not in globals()
]

if faltantes:
    raise RuntimeError(
        "Faltan objetos necesarios. Ejecuta primero las celdas "
        f"de Feature Extraction correspondientes:\n{faltantes}"
    )

columnas_preprocesamiento_candidatas = [
    "comentario_limpio",
    "comentario_limpio_sin_tildes",
    "texto_v1_limpio",
    "texto_v2_sin_tildes",
    "texto_v3_sin_stopwords",
    "texto_v4_stemming",
    "texto_v5_lemma",
    "tokens",
    "tokens_sin_stopwords",
    "tokens_stem",
    "tokens_lemma",
    "texto_para_ngrams",
    "tokens_para_word2vec"
]

columnas_preprocesamiento = [
    columna
    for columna in columnas_preprocesamiento_candidatas
    if columna in df_features_base.columns
]

df_textos_para_exportar = df_features_base[
    ["idx_feature_matrix"] + columnas_preprocesamiento
].copy()

def convertir_listas_a_texto(df_base):
    df_salida = df_base.copy()

    for columna in df_salida.columns:
        tiene_listas = df_salida[columna].apply(
            lambda valor: isinstance(valor, (list, tuple, set, np.ndarray))
        ).any()

        if tiene_listas:
            df_salida[columna] = df_salida[columna].apply(
                lambda valor: " ".join(map(str, valor))
                if isinstance(valor, (list, tuple, set, np.ndarray))
                else valor
            )

    return df_salida

columnas_extra_tfidf = [
    columna
    for columna in columnas_preprocesamiento
    if columna not in df_ngram_version.columns
]

df_etiquetado_tfidf = df_ngram_version.merge(
    df_textos_para_exportar[
        ["idx_feature_matrix"] + columnas_extra_tfidf
    ],
    on="idx_feature_matrix",
    how="left",
    validate="one_to_one"
)

columnas_vectores_w2v = [
    columna
    for columna in df_word2vec_version.columns
    if columna.startswith("w2v_")
]

df_word2vec_sin_vectores = df_word2vec_version.drop(
    columns=columnas_vectores_w2v
).copy()

columnas_extra_w2v = [
    columna
    for columna in columnas_preprocesamiento
    if columna not in df_word2vec_sin_vectores.columns
]

df_etiquetado_word2vec = df_word2vec_sin_vectores.merge(
    df_textos_para_exportar[
        ["idx_feature_matrix"] + columnas_extra_w2v
    ],
    on="idx_feature_matrix",
    how="left",
    validate="one_to_one"
)

columnas_etiquetado = {
    "estado_etiquetado": "pendiente",
    "etiqueta_manual": "",
    "observaciones_etiquetado": ""
}

for columna, valor_inicial in columnas_etiquetado.items():
    df_etiquetado_tfidf[columna] = valor_inicial
    df_etiquetado_word2vec[columna] = valor_inicial

orden_preferido = [
    "idx_feature_matrix",
    "ResponseId",
    "pregunta_origen",
    "comentario_original",
    "comentario_limpio",
    "texto_para_ngrams",
    "tokens_para_word2vec",
    "texto_v1_limpio",
    "texto_v2_sin_tildes",
    "texto_v3_sin_stopwords",
    "texto_v4_stemming",
    "texto_v5_lemma",
    "tokens",
    "tokens_sin_stopwords",
    "tokens_stem",
    "tokens_lemma",
    "feature_extraction_method",
    "feature_extraction_detail",
    "texto_base_usado",
    "tokens_base_usados",
    "estado_etiquetado",
    "etiqueta_manual",
    "observaciones_etiquetado"
]

def ordenar_columnas(df_base):
    columnas_existentes_ordenadas = [
        columna for columna in orden_preferido
        if columna in df_base.columns
    ]

    columnas_restantes = [
        columna for columna in df_base.columns
        if columna not in columnas_existentes_ordenadas
    ]

    return df_base[
        columnas_existentes_ordenadas + columnas_restantes
    ].copy()

df_etiquetado_tfidf = ordenar_columnas(df_etiquetado_tfidf)
df_etiquetado_word2vec = ordenar_columnas(df_etiquetado_word2vec)

df_etiquetado_tfidf = convertir_listas_a_texto(df_etiquetado_tfidf)
df_etiquetado_word2vec = convertir_listas_a_texto(df_etiquetado_word2vec)

RUTA_EXPORTACION = Path("bases_para_etiquetar")
RUTA_EXPORTACION.mkdir(parents=True, exist_ok=True)

ruta_tfidf = RUTA_EXPORTACION / "base_etiquetado_tfidf_ngrams.xlsx"
ruta_word2vec = RUTA_EXPORTACION / "base_etiquetado_word2vec.xlsx"

with pd.ExcelWriter(ruta_tfidf, engine="openpyxl") as writer:
    df_etiquetado_tfidf.to_excel(
        writer,
        sheet_name="Etiquetado_TFIDF",
        index=False
    )

    hoja = writer.sheets["Etiquetado_TFIDF"]
    hoja.freeze_panes = "A2"
    hoja.auto_filter.ref = hoja.dimensions

with pd.ExcelWriter(ruta_word2vec, engine="openpyxl") as writer:
    df_etiquetado_word2vec.to_excel(
        writer,
        sheet_name="Etiquetado_Word2Vec",
        index=False
    )

    hoja = writer.sheets["Etiquetado_Word2Vec"]
    hoja.freeze_panes = "A2"
    hoja.auto_filter.ref = hoja.dimensions

print("Excel TF-IDF creado en:")
print(ruta_tfidf.resolve())

print("\nExcel Word2Vec creado en:")
print(ruta_word2vec.resolve())

print("\nResumen:")
print("Comentarios TF-IDF:", len(df_etiquetado_tfidf))
print("Comentarios Word2Vec:", len(df_etiquetado_word2vec))
print("Columnas TF-IDF:", df_etiquetado_tfidf.shape[1])
print("Columnas Word2Vec:", df_etiquetado_word2vec.shape[1])

Excel TF-IDF creado en:
H:\Mi unidad\Proyecto ML Universidad\bases_para_etiquetar\base_etiquetado_tfidf_ngrams.xlsx

Excel Word2Vec creado en:
H:\Mi unidad\Proyecto ML Universidad\bases_para_etiquetar\base_etiquetado_word2vec.xlsx

Resumen:
Comentarios TF-IDF: 6119
Comentarios Word2Vec: 6119
Columnas TF-IDF: 23
Columnas Word2Vec: 23


### 6.2 Cruce de etiquetas con las representaciones

Se leen los archivos Excel etiquetados, se validan las columnas `cat_` y se unen las etiquetas con las representaciones TF-IDF y Word2Vec mediante `idx_feature_matrix`.

In [31]:
import pandas as pd
import numpy as np

RUTA_TFIDF_ETIQUETADO = "base_etiquetado_tfidf_ngrams.xlsx"
RUTA_WORD2VEC_ETIQUETADO = "base_etiquetado_word2vec.xlsx"

HOJA_TFIDF = "Etiquetado_TFIDF"
HOJA_WORD2VEC = "Etiquetado_Word2Vec"

objetos_requeridos = [
    "df_ngram_version",
    "df_word2vec_version",
    "X_tfidf_ngrams",
    "X_word2vec"
]

faltantes = [
    objeto for objeto in objetos_requeridos
    if objeto not in globals()
]

if faltantes:
    raise RuntimeError(
        "Faltan objetos del notebook. Ejecuta primero las celdas "
        f"de Feature Extraction:\n{faltantes}"
    )

df_etiquetas_tfidf = pd.read_excel(
    RUTA_TFIDF_ETIQUETADO,
    sheet_name=HOJA_TFIDF
)

df_etiquetas_word2vec = pd.read_excel(
    RUTA_WORD2VEC_ETIQUETADO,
    sheet_name=HOJA_WORD2VEC
)

cat_cols_tfidf = [
    columna
    for columna in df_etiquetas_tfidf.columns
    if str(columna).startswith("cat_")
]

cat_cols_word2vec = [
    columna
    for columna in df_etiquetas_word2vec.columns
    if str(columna).startswith("cat_")
]

if not cat_cols_tfidf:
    raise ValueError(
        "No se encontraron columnas que comiencen con 'cat_' "
        "en el Excel TF-IDF."
    )

if set(cat_cols_tfidf) != set(cat_cols_word2vec):
    raise ValueError(
        "Los dos Excel no tienen las mismas columnas cat_.\n"
        f"TF-IDF: {cat_cols_tfidf}\n"
        f"Word2Vec: {cat_cols_word2vec}"
    )

CAT_COLS = cat_cols_tfidf

def validar_archivo_etiquetado(df, nombre_archivo, columnas_categoria):

    if "idx_feature_matrix" not in df.columns:
        raise ValueError(
            f"{nombre_archivo} no contiene idx_feature_matrix."
        )

    if df["idx_feature_matrix"].duplicated().any():
        ejemplos = df.loc[
            df["idx_feature_matrix"].duplicated(keep=False),
            "idx_feature_matrix"
        ].head(10).tolist()

        raise ValueError(
            f"{nombre_archivo} tiene idx_feature_matrix duplicados. "
            f"Ejemplos: {ejemplos}"
        )

    for columna in columnas_categoria:
        df[columna] = pd.to_numeric(
            df[columna],
            errors="coerce"
        )

        valores_invalidos = df.loc[
            df[columna].notna() & ~df[columna].isin([0, 1]),
            columna
        ].unique()

        if len(valores_invalidos) > 0:
            raise ValueError(
                f"Valores inválidos en {nombre_archivo}, columna "
                f"'{columna}': {valores_invalidos}. "
                "Solo se permiten 0, 1 o vacío."
            )

    return df

df_etiquetas_tfidf = validar_archivo_etiquetado(
    df_etiquetas_tfidf,
    "Excel TF-IDF",
    CAT_COLS
)

df_etiquetas_word2vec = validar_archivo_etiquetado(
    df_etiquetas_word2vec,
    "Excel Word2Vec",
    CAT_COLS
)

etiquetas_tfidf_comparacion = (
    df_etiquetas_tfidf
    .set_index("idx_feature_matrix")[CAT_COLS]
    .sort_index()
)

etiquetas_word2vec_comparacion = (
    df_etiquetas_word2vec
    .set_index("idx_feature_matrix")[CAT_COLS]
    .sort_index()
)

if not etiquetas_tfidf_comparacion.equals(
    etiquetas_word2vec_comparacion
):
    raise ValueError(
        "Las etiquetas cat_ de TF-IDF y Word2Vec no coinciden. "
        "Revisa los dos Excel antes de entrenar."
    )

df_etiquetas_finales = df_etiquetas_tfidf[
    ["idx_feature_matrix"] + CAT_COLS
].copy()

mask_etiquetado = df_etiquetas_finales[CAT_COLS].notna().all(axis=1)

if mask_etiquetado.sum() == 0:
    raise ValueError(
        "No existen filas con todas las categorías cat_ completas."
    )

df_etiquetas_finales["estado_etiquetado"] = np.where(
    mask_etiquetado,
    "Listo",
    "pendiente"
)

def validar_version_feature(df, nombre):

    if "idx_feature_matrix" not in df.columns:
        raise ValueError(
            f"{nombre} no contiene idx_feature_matrix."
        )

    if df["idx_feature_matrix"].duplicated().any():
        raise ValueError(
            f"{nombre} contiene idx_feature_matrix duplicados."
        )

validar_version_feature(df_ngram_version, "df_ngram_version")
validar_version_feature(df_word2vec_version, "df_word2vec_version")

if len(df_ngram_version) != X_tfidf_ngrams.shape[0]:
    raise ValueError(
        "df_ngram_version y X_tfidf_ngrams no tienen "
        "la misma cantidad de filas."
    )

if len(df_word2vec_version) != X_word2vec.shape[0]:
    raise ValueError(
        "df_word2vec_version y X_word2vec no tienen "
        "la misma cantidad de filas."
    )

columnas_a_reemplazar = CAT_COLS + ["estado_etiquetado"]

df_ngram_sin_etiquetas = df_ngram_version.drop(
    columns=columnas_a_reemplazar,
    errors="ignore"
).copy()

df_word2vec_sin_etiquetas = df_word2vec_version.drop(
    columns=columnas_a_reemplazar,
    errors="ignore"
).copy()

df_ngram_con_etiquetas = df_ngram_sin_etiquetas.merge(
    df_etiquetas_finales,
    on="idx_feature_matrix",
    how="left",
    validate="one_to_one",
    sort=False
)

df_word2vec_con_etiquetas = df_word2vec_sin_etiquetas.merge(
    df_etiquetas_finales,
    on="idx_feature_matrix",
    how="left",
    validate="one_to_one",
    sort=False
)

mask_tfidf = (
    df_ngram_con_etiquetas["estado_etiquetado"]
    .eq("Listo")
    .to_numpy()
)

mask_word2vec = (
    df_word2vec_con_etiquetas["estado_etiquetado"]
    .eq("Listo")
    .to_numpy()
)

df_tfidf_entrenamiento = (
    df_ngram_con_etiquetas
    .loc[mask_tfidf]
    .reset_index(drop=True)
)

df_word2vec_entrenamiento = (
    df_word2vec_con_etiquetas
    .loc[mask_word2vec]
    .reset_index(drop=True)
)

X_tfidf_entrenamiento = X_tfidf_ngrams[mask_tfidf]
X_word2vec_entrenamiento = X_word2vec[mask_word2vec]

Y_tfidf_entrenamiento = (
    df_tfidf_entrenamiento[CAT_COLS]
    .astype(int)
    .copy()
)

Y_word2vec_entrenamiento = (
    df_word2vec_entrenamiento[CAT_COLS]
    .astype(int)
    .copy()
)

print("=" * 65)
print("CRUCE FINALIZADO CORRECTAMENTE")
print("=" * 65)

print(f"\nCategorías detectadas ({len(CAT_COLS)}):")
for categoria in CAT_COLS:
    print(f" - {categoria}")

print(f"\nTotal universo de comentarios: {len(df_etiquetas_finales):,}")
print(f"Comentarios listos para entrenar: {mask_etiquetado.sum():,}")

print("\nShape TF-IDF para entrenamiento:")
print(X_tfidf_entrenamiento.shape)

print("\nShape Word2Vec para entrenamiento:")
print(X_word2vec_entrenamiento.shape)

print("\nDistribución de positivos por categoría:")
display(
    Y_tfidf_entrenamiento
    .sum()
    .sort_values(ascending=False)
    .to_frame("comentarios_positivos")
)

CRUCE FINALIZADO CORRECTAMENTE

Categorías detectadas (5):
 - cat_Pago_monto_y_calculo
 - cat_Tiempos_y_oportunidad
 - cat_Informacion_y_comunicacion
 - cat_Atencion_y_gestion_del_servicio
 - cat_Satisfaccion_o_comentario_general

Total universo de comentarios: 6,119
Comentarios listos para entrenar: 1,500

Shape TF-IDF para entrenamiento:
(1500, 7110)

Shape Word2Vec para entrenamiento:
(1500, 100)

Distribución de positivos por categoría:


,comentarios_positivos
cat_Pago_monto_y_calculo,343
cat_Tiempos_y_oportunidad,339
cat_Informacion_y_comunicacion,327
cat_Atencion_y_gestion_del_servicio,320
cat_Satisfaccion_o_comentario_general,300


## Etapa 7: Modelamiento KNN

Se instalan y cargan las dependencias necesarias para estratificación multietiqueta. Las etapas siguientes utilizan una misma partición de entrenamiento, validación y prueba para comparar de forma justa los diez experimentos.

In [32]:
%pip install iterative-stratification

Note: you may need to restart the kernel to use updated packages.


### Etapa KNN 1: Configuración y base de modelamiento

Se definen los parámetros reproducibles, las cinco versiones de texto, las categorías objetivo y la base final de comentarios etiquetados.

In [33]:
import random
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    hamming_loss,
    classification_report,
    multilabel_confusion_matrix
)

from gensim.models import Word2Vec
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

SEED = 42

TEST_SIZE = 0.15
VALIDATION_SIZE = 0.15

N_JOBS_KNN = -1

W2V_VECTOR_SIZE = 100
W2V_WINDOW = 5
W2V_MIN_COUNT = 2
W2V_SG = 1
W2V_EPOCHS = 5
W2V_WORKERS = 1

random.seed(SEED)
np.random.seed(SEED)

objetos_requeridos = [
    "df_features_base",
    "df_etiquetas_finales",
    "CAT_COLS"
]

faltantes = [
    objeto
    for objeto in objetos_requeridos
    if objeto not in globals()
]

if faltantes:
    raise RuntimeError(
        "Faltan objetos necesarios del notebook: "
        f"{faltantes}. Ejecuta primero la celda de cruce de etiquetas."
    )

VERSIONES_EXPERIMENTO = [
    {
        "nombre_version": "texto_v1_limpio",
        "columna_texto": "texto_v1_limpio",
        "columna_tokens": "tokens"
    },
    {
        "nombre_version": "texto_v2_sin_tildes",
        "columna_texto": "texto_v2_sin_tildes",
        "columna_tokens": "tokens_sin_tildes"
    },
    {
        "nombre_version": "texto_v3_sin_stopwords",
        "columna_texto": "texto_v3_sin_stopwords",
        "columna_tokens": "tokens_sin_stopwords"
    },
    {
        "nombre_version": "texto_v4_stemming",
        "columna_texto": "texto_v4_stemming",
        "columna_tokens": "tokens_stem"
    },
    {
        "nombre_version": "texto_v5_lemma",
        "columna_texto": "texto_v5_lemma",
        "columna_tokens": "tokens_lemma"
    }
]

columnas_versiones_requeridas = []

for version in VERSIONES_EXPERIMENTO:
    columnas_versiones_requeridas.extend([
        version["columna_texto"],
        version["columna_tokens"]
    ])

columnas_faltantes = [
    columna
    for columna in columnas_versiones_requeridas
    if columna not in df_features_base.columns
]

if columnas_faltantes:
    raise ValueError(
        "No se pueden ejecutar los 10 experimentos porque faltan "
        f"estas columnas en df_features_base:\n{columnas_faltantes}"
    )

df_base_modelado = df_features_base.copy()
df_etiquetas_modelado = df_etiquetas_finales.copy()

df_base_modelado["idx_feature_matrix"] = pd.to_numeric(
    df_base_modelado["idx_feature_matrix"],
    errors="raise"
).astype(int)

df_etiquetas_modelado["idx_feature_matrix"] = pd.to_numeric(
    df_etiquetas_modelado["idx_feature_matrix"],
    errors="raise"
).astype(int)

df_modelado = df_base_modelado.merge(
    df_etiquetas_modelado,
    on="idx_feature_matrix",
    how="left",
    validate="one_to_one"
)

df_modelado = (
    df_modelado
    .loc[df_modelado["estado_etiquetado"].eq("Listo")]
    .copy()
    .sort_values("idx_feature_matrix")
    .reset_index(drop=True)
)

if df_modelado.empty:
    raise ValueError(
        "No existen comentarios con estado_etiquetado = 'Listo'."
    )

if df_modelado[CAT_COLS].isna().any().any():
    raise ValueError(
        "Existen valores vacíos en columnas cat_ dentro de los comentarios Listo."
    )

df_modelado[CAT_COLS] = df_modelado[CAT_COLS].astype(int)

Y_modelado = df_modelado[CAT_COLS].to_numpy(dtype=int)

if (Y_modelado.sum(axis=0) == 0).any():
    categorias_sin_positivos = [
        CAT_COLS[i]
        for i, cantidad in enumerate(Y_modelado.sum(axis=0))
        if cantidad == 0
    ]

    raise ValueError(
        "Hay categorías sin ejemplos positivos:\n"
        f"{categorias_sin_positivos}"
    )

df_distribucion_etiquetas = pd.DataFrame({
    "categoria": CAT_COLS,
    "comentarios_positivos": Y_modelado.sum(axis=0),
    "porcentaje_positivo": Y_modelado.mean(axis=0) * 100
}).sort_values(
    "comentarios_positivos",
    ascending=False
).reset_index(drop=True)

print("=" * 70)
print("BASE LISTA PARA EXPERIMENTOS KNN")
print("=" * 70)
print(f"Comentarios etiquetados disponibles: {len(df_modelado):,}")
print(f"Categorías objetivo: {len(CAT_COLS)}")
print(f"Experimentos esperados: {len(VERSIONES_EXPERIMENTO) * 2}")

display(df_distribucion_etiquetas)

BASE LISTA PARA EXPERIMENTOS KNN
Comentarios etiquetados disponibles: 1,500
Categorías objetivo: 5
Experimentos esperados: 10


,categoria,comentarios_positivos,porcentaje_positivo
0,cat_Pago_monto_y_calculo,343,22.866667
1,cat_Tiempos_y_oportunidad,339,22.600000
2,cat_Informacion_y_comunicacion,327,21.800000
3,cat_Atencion_y_gestion_del_servicio,320,21.333333
4,cat_Satisfaccion_o_comentario_general,300,20.000000


### Etapa KNN 2: División train, validation y test

Se crea una división multietiqueta común para todos los experimentos, preservando en lo posible la distribución de cada categoría.

In [34]:
indices_totales = np.arange(len(df_modelado))
X_dummy = np.zeros((len(df_modelado), 1))

splitter_test = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=SEED
)

indices_train_val, indices_test = next(
    splitter_test.split(X_dummy, Y_modelado)
)

proporcion_validacion_relativa = VALIDATION_SIZE / (1 - TEST_SIZE)

splitter_validacion = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=proporcion_validacion_relativa,
    random_state=SEED
)

indices_train_relativos, indices_val_relativos = next(
    splitter_validacion.split(
        np.zeros((len(indices_train_val), 1)),
        Y_modelado[indices_train_val]
    )
)

indices_train = indices_train_val[indices_train_relativos]
indices_val = indices_train_val[indices_val_relativos]

assert len(set(indices_train) & set(indices_val)) == 0
assert len(set(indices_train) & set(indices_test)) == 0
assert len(set(indices_val) & set(indices_test)) == 0

assert (
    len(indices_train) +
    len(indices_val) +
    len(indices_test)
) == len(df_modelado)

Y_train = Y_modelado[indices_train]
Y_val = Y_modelado[indices_val]
Y_test = Y_modelado[indices_test]

Y_train_val = Y_modelado[indices_train_val]

df_modelado["particion"] = "sin_asignar"
df_modelado.loc[indices_train, "particion"] = "train"
df_modelado.loc[indices_val, "particion"] = "validation"
df_modelado.loc[indices_test, "particion"] = "test"

def resumir_particion(nombre_particion, matriz_y):
    return pd.DataFrame({
        "particion": nombre_particion,
        "categoria": CAT_COLS,
        "comentarios_positivos": matriz_y.sum(axis=0),
        "porcentaje_positivo": matriz_y.mean(axis=0) * 100
    })

df_resumen_particiones = pd.concat([
    resumir_particion("train", Y_train),
    resumir_particion("validation", Y_val),
    resumir_particion("test", Y_test)
], ignore_index=True)

print("=" * 70)
print("DIVISIÓN MULTIETIQUETA CREADA")
print("=" * 70)
print(f"Train:      {len(indices_train):,} comentarios")
print(f"Validation: {len(indices_val):,} comentarios")
print(f"Test:       {len(indices_test):,} comentarios")

display(
    df_resumen_particiones.pivot(
        index="categoria",
        columns="particion",
        values="comentarios_positivos"
    )
)

DIVISIÓN MULTIETIQUETA CREADA
Train:      1,051 comentarios
Validation: 226 comentarios
Test:       223 comentarios


particion,test,train,validation
categoria,,,
cat_Atencion_y_gestion_del_servicio,48,224,48
cat_Informacion_y_comunicacion,49,229,49
cat_Pago_monto_y_calculo,51,240,52
cat_Satisfaccion_o_comentario_general,45,210,45
cat_Tiempos_y_oportunidad,51,237,51


### Etapa KNN 3: Funciones y selección de hiperparámetros

Se definen las grillas de hiperparámetros y las funciones reutilizables para construir representaciones, entrenar KNN y calcular métricas multietiqueta.

In [35]:
K_VALUES = [3, 5, 7, 9, 11]
WEIGHTS_VALUES = ["uniform", "distance"]

ALGORITMOS_WORD2VEC = [
    "auto",
    "ball_tree",
    "kd_tree",
    "brute"
]

def crear_candidatos_tfidf():
    candidatos = []

    for k in K_VALUES:
        for weights in WEIGHTS_VALUES:

            candidatos.append({
                "n_neighbors": k,
                "weights": weights,
                "metric": "cosine",
                "p": None,
                "algorithm": "brute"
            })

            candidatos.append({
                "n_neighbors": k,
                "weights": weights,
                "metric": "minkowski",
                "p": 1,
                "algorithm": "brute"
            })

            candidatos.append({
                "n_neighbors": k,
                "weights": weights,
                "metric": "minkowski",
                "p": 2,
                "algorithm": "brute"
            })

    return candidatos

def crear_candidatos_word2vec():
    candidatos = []

    for k in K_VALUES:
        for weights in WEIGHTS_VALUES:

            candidatos.append({
                "n_neighbors": k,
                "weights": weights,
                "metric": "cosine",
                "p": None,
                "algorithm": "brute"
            })

            for p in [1, 2]:
                for algorithm in ALGORITMOS_WORD2VEC:
                    candidatos.append({
                        "n_neighbors": k,
                        "weights": weights,
                        "metric": "minkowski",
                        "p": p,
                        "algorithm": algorithm
                    })

    return candidatos

CANDIDATOS_TFIDF = crear_candidatos_tfidf()
CANDIDATOS_WORD2VEC = crear_candidatos_word2vec()

print("Configuraciones TF-IDF:", len(CANDIDATOS_TFIDF))
print("Configuraciones Word2Vec:", len(CANDIDATOS_WORD2VEC))

def texto_seguro(valor):
    """Convierte valores vacíos a texto vacío."""
    if pd.isna(valor):
        return ""
    return str(valor)

def normalizar_tokens(valor):
    """Normaliza una secuencia de tokens."""

    if isinstance(valor, (list, tuple, np.ndarray, pd.Series)):
        return [
            str(token).strip()
            for token in list(valor)
            if token is not None
            and str(token).strip() != ""
            and str(token).lower() != "nan"
        ]

    if pd.isna(valor):
        return []

    return str(valor).split()

def crear_knn(parametros):
    """Construye un clasificador KNN."""

    argumentos = {
        "n_neighbors": int(parametros["n_neighbors"]),
        "weights": parametros["weights"],
        "metric": parametros["metric"],
        "algorithm": parametros["algorithm"],
        "n_jobs": N_JOBS_KNN
    }

    if parametros["metric"] == "minkowski":
        argumentos["p"] = int(parametros["p"])

    return KNeighborsClassifier(**argumentos)

def calcular_metricas_multietiqueta(y_real, y_predicho):
    """Calcula métricas globales multietiqueta."""

    return {
        "accuracy_exacta": accuracy_score(y_real, y_predicho),
        "precision_macro": precision_score(
            y_real,
            y_predicho,
            average="macro",
            zero_division=0
        ),
        "recall_macro": recall_score(
            y_real,
            y_predicho,
            average="macro",
            zero_division=0
        ),
        "f1_macro": f1_score(
            y_real,
            y_predicho,
            average="macro",
            zero_division=0
        ),
        "precision_micro": precision_score(
            y_real,
            y_predicho,
            average="micro",
            zero_division=0
        ),
        "recall_micro": recall_score(
            y_real,
            y_predicho,
            average="micro",
            zero_division=0
        ),
        "f1_micro": f1_score(
            y_real,
            y_predicho,
            average="micro",
            zero_division=0
        ),
        "f1_weighted": f1_score(
            y_real,
            y_predicho,
            average="weighted",
            zero_division=0
        ),
        "hamming_loss": hamming_loss(y_real, y_predicho)
    }

def construir_tfidf(textos_ajuste, textos_evaluacion):
    """Ajusta y transforma TF-IDF sin fuga de información."""

    inicio = time.perf_counter()

    vectorizador = TfidfVectorizer(
        lowercase=False,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        dtype=np.float32
    )

    X_ajuste = vectorizador.fit_transform(textos_ajuste)
    X_evaluacion = vectorizador.transform(textos_evaluacion)

    tiempo_representacion = time.perf_counter() - inicio

    documentos_sin_vector = int(
        np.sum(X_evaluacion.getnnz(axis=1) == 0)
    )

    metadatos = {
        "n_features": X_ajuste.shape[1],
        "vocabulario_tfidf": len(vectorizador.vocabulary_),
        "documentos_sin_vector_evaluacion": documentos_sin_vector,
        "tiempo_representacion_seg": tiempo_representacion
    }

    return vectorizador, X_ajuste, X_evaluacion, metadatos

def vectorizar_documentos_word2vec(tokens_documentos, modelo_w2v):
    """Convierte comentarios en vectores Word2Vec promedio."""

    matriz = np.zeros(
        (len(tokens_documentos), modelo_w2v.vector_size),
        dtype=np.float32
    )

    documentos_sin_vector = 0

    for indice, tokens in enumerate(tokens_documentos):
        tokens_limpios = normalizar_tokens(tokens)

        vectores = [
            modelo_w2v.wv[token]
            for token in tokens_limpios
            if token in modelo_w2v.wv
        ]

        if len(vectores) == 0:
            documentos_sin_vector += 1
            continue

        matriz[indice] = np.mean(vectores, axis=0)

    return matriz, documentos_sin_vector

def construir_word2vec(tokens_ajuste, tokens_evaluacion):
    """Entrena y aplica Word2Vec sin fuga de información."""

    inicio = time.perf_counter()

    corpus_ajuste = [
        normalizar_tokens(tokens)
        for tokens in tokens_ajuste
    ]

    corpus_ajuste = [
        tokens
        for tokens in corpus_ajuste
        if len(tokens) > 0
    ]

    if len(corpus_ajuste) == 0:
        raise ValueError(
            "No hay documentos con tokens para entrenar Word2Vec."
        )

    modelo_w2v = Word2Vec(
        sentences=corpus_ajuste,
        vector_size=W2V_VECTOR_SIZE,
        window=W2V_WINDOW,
        min_count=W2V_MIN_COUNT,
        sg=W2V_SG,
        epochs=W2V_EPOCHS,
        workers=W2V_WORKERS,
        seed=SEED
    )

    X_ajuste, documentos_sin_vector_ajuste = (
        vectorizar_documentos_word2vec(tokens_ajuste, modelo_w2v)
    )

    X_evaluacion, documentos_sin_vector_evaluacion = (
        vectorizar_documentos_word2vec(tokens_evaluacion, modelo_w2v)
    )

    tiempo_representacion = time.perf_counter() - inicio

    metadatos = {
        "n_features": modelo_w2v.vector_size,
        "vocabulario_word2vec": len(modelo_w2v.wv.index_to_key),
        "documentos_sin_vector_ajuste": documentos_sin_vector_ajuste,
        "documentos_sin_vector_evaluacion": documentos_sin_vector_evaluacion,
        "tiempo_representacion_seg": tiempo_representacion
    }

    return modelo_w2v, X_ajuste, X_evaluacion, metadatos

def evaluar_hiperparametros_knn(
    X_train_local,
    Y_train_local,
    X_val_local,
    Y_val_local,
    candidatos,
    nombre_modelo,
    representacion,
    columna_texto
):
    """Evalúa configuraciones KNN sobre validación."""

    resultados = []

    inicio_busqueda = time.perf_counter()

    for numero_candidato, parametros in enumerate(candidatos, start=1):

        modelo_knn = crear_knn(parametros)

        inicio_fit = time.perf_counter()
        modelo_knn.fit(X_train_local, Y_train_local)
        tiempo_fit = time.perf_counter() - inicio_fit

        inicio_prediccion = time.perf_counter()
        Y_val_pred = modelo_knn.predict(X_val_local)
        tiempo_prediccion = time.perf_counter() - inicio_prediccion

        metricas = calcular_metricas_multietiqueta(
            Y_val_local,
            Y_val_pred
        )

        resultados.append({
            "nombre_modelo": nombre_modelo,
            "representacion": representacion,
            "columna_texto": columna_texto,
            "numero_candidato": numero_candidato,
            "n_neighbors": parametros["n_neighbors"],
            "weights": parametros["weights"],
            "metric": parametros["metric"],
            "p": parametros["p"],
            "algorithm": parametros["algorithm"],
            "tiempo_fit_validacion_seg": tiempo_fit,
            "tiempo_prediccion_validacion_seg": tiempo_prediccion,
            "tiempo_prediccion_validacion_ms_por_comentario": (
                tiempo_prediccion / len(Y_val_local)
            ) * 1000,
            **metricas
        })

    tiempo_busqueda = time.perf_counter() - inicio_busqueda

    df_resultados = pd.DataFrame(resultados)

    df_resultados = df_resultados.sort_values(
        by=[
            "f1_macro",
            "f1_micro",
            "precision_macro",
            "tiempo_prediccion_validacion_seg"
        ],
        ascending=[
            False,
            False,
            False,
            True
        ]
    ).reset_index(drop=True)

    return df_resultados, tiempo_busqueda

def crear_reporte_clasificacion(y_real, y_predicho, nombre_modelo):
    reporte = classification_report(
        y_real,
        y_predicho,
        target_names=CAT_COLS,
        output_dict=True,
        zero_division=0
    )

    df_reporte = (
        pd.DataFrame(reporte)
        .T
        .reset_index()
        .rename(columns={"index": "clase"})
    )

    df_reporte.insert(0, "nombre_modelo", nombre_modelo)

    return df_reporte

def crear_tabla_matrices_confusion(y_real, y_predicho, nombre_modelo):
    matrices = multilabel_confusion_matrix(y_real, y_predicho)

    filas = []

    for indice_categoria, categoria in enumerate(CAT_COLS):
        tn, fp, fn, tp = matrices[indice_categoria].ravel()

        filas.append({
            "nombre_modelo": nombre_modelo,
            "categoria": categoria,
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "TP": tp
        })

    return pd.DataFrame(filas)

Configuraciones TF-IDF: 30
Configuraciones Word2Vec: 90


### Etapa KNN 4: Entrenamiento y evaluación de los diez experimentos

Para cada versión de texto se evalúan TF-IDF con n-gramas y Word2Vec. Los hiperparámetros se seleccionan con validación y la evaluación final se realiza sobre el conjunto de prueba.

In [36]:
resultados_finales = []
resultados_validacion_todos = []

reportes_clasificacion = {}
matrices_confusion = {}
predicciones_test_modelos = {}
modelos_finales = {}
errores_experimentos = []

total_experimentos = len(VERSIONES_EXPERIMENTO) * 2
contador_experimento = 0

for configuracion in VERSIONES_EXPERIMENTO:

    nombre_version = configuracion["nombre_version"]
    columna_texto = configuracion["columna_texto"]
    columna_tokens = configuracion["columna_tokens"]

    for representacion in ["TFIDF_NGRAMS", "WORD2VEC"]:

        contador_experimento += 1

        nombre_modelo = (
            f"KNN__{representacion}__{nombre_version}"
        )

        print("\n" + "=" * 85)
        print(
            f"EXPERIMENTO {contador_experimento}/{total_experimentos}: "
            f"{nombre_modelo}"
        )
        print("=" * 85)

        inicio_experimento = time.perf_counter()

        try:

            textos_train = (
                df_modelado
                .iloc[indices_train][columna_texto]
                .apply(texto_seguro)
                .to_numpy()
            )

            textos_val = (
                df_modelado
                .iloc[indices_val][columna_texto]
                .apply(texto_seguro)
                .to_numpy()
            )

            textos_train_val = (
                df_modelado
                .iloc[indices_train_val][columna_texto]
                .apply(texto_seguro)
                .to_numpy()
            )

            textos_test = (
                df_modelado
                .iloc[indices_test][columna_texto]
                .apply(texto_seguro)
                .to_numpy()
            )

            tokens_train = (
                df_modelado
                .iloc[indices_train][columna_tokens]
                .tolist()
            )

            tokens_val = (
                df_modelado
                .iloc[indices_val][columna_tokens]
                .tolist()
            )

            tokens_train_val = (
                df_modelado
                .iloc[indices_train_val][columna_tokens]
                .tolist()
            )

            tokens_test = (
                df_modelado
                .iloc[indices_test][columna_tokens]
                .tolist()
            )

            if representacion == "TFIDF_NGRAMS":

                (
                    vectorizador_validacion,
                    X_train_actual,
                    X_val_actual,
                    metadatos_validacion
                ) = construir_tfidf(
                    textos_ajuste=textos_train,
                    textos_evaluacion=textos_val
                )

                candidatos_actuales = CANDIDATOS_TFIDF

                (
                    df_validacion_modelo,
                    tiempo_busqueda_hiperparametros
                ) = evaluar_hiperparametros_knn(
                    X_train_local=X_train_actual,
                    Y_train_local=Y_train,
                    X_val_local=X_val_actual,
                    Y_val_local=Y_val,
                    candidatos=candidatos_actuales,
                    nombre_modelo=nombre_modelo,
                    representacion=representacion,
                    columna_texto=columna_texto
                )

                mejor_configuracion = (
                    df_validacion_modelo
                    .iloc[0]
                    .to_dict()
                )

                (
                    vectorizador_final,
                    X_train_val_actual,
                    X_test_actual,
                    metadatos_finales_representacion
                ) = construir_tfidf(
                    textos_ajuste=textos_train_val,
                    textos_evaluacion=textos_test
                )

                objeto_representacion_final = vectorizador_final

            else:

                (
                    modelo_w2v_validacion,
                    X_train_actual,
                    X_val_actual,
                    metadatos_validacion
                ) = construir_word2vec(
                    tokens_ajuste=tokens_train,
                    tokens_evaluacion=tokens_val
                )

                candidatos_actuales = CANDIDATOS_WORD2VEC

                (
                    df_validacion_modelo,
                    tiempo_busqueda_hiperparametros
                ) = evaluar_hiperparametros_knn(
                    X_train_local=X_train_actual,
                    Y_train_local=Y_train,
                    X_val_local=X_val_actual,
                    Y_val_local=Y_val,
                    candidatos=candidatos_actuales,
                    nombre_modelo=nombre_modelo,
                    representacion=representacion,
                    columna_texto=columna_texto
                )

                mejor_configuracion = (
                    df_validacion_modelo
                    .iloc[0]
                    .to_dict()
                )

                (
                    modelo_w2v_final,
                    X_train_val_actual,
                    X_test_actual,
                    metadatos_finales_representacion
                ) = construir_word2vec(
                    tokens_ajuste=tokens_train_val,
                    tokens_evaluacion=tokens_test
                )

                objeto_representacion_final = modelo_w2v_final

            parametros_ganadores = {
                "n_neighbors": int(mejor_configuracion["n_neighbors"]),
                "weights": mejor_configuracion["weights"],
                "metric": mejor_configuracion["metric"],
                "algorithm": mejor_configuracion["algorithm"],
                "p": (
                    None
                    if pd.isna(mejor_configuracion["p"])
                    else int(mejor_configuracion["p"])
                )
            }

            modelo_final_knn = crear_knn(parametros_ganadores)

            inicio_entrenamiento_final = time.perf_counter()

            modelo_final_knn.fit(
                X_train_val_actual,
                Y_train_val
            )

            tiempo_entrenamiento_final = (
                time.perf_counter() - inicio_entrenamiento_final
            )

            inicio_clasificacion_test = time.perf_counter()

            Y_test_pred = modelo_final_knn.predict(
                X_test_actual
            )

            tiempo_clasificacion_test = (
                time.perf_counter() - inicio_clasificacion_test
            )

            metricas_test = calcular_metricas_multietiqueta(
                Y_test,
                Y_test_pred
            )

            df_validacion_modelo["ranking_validacion"] = (
                np.arange(1, len(df_validacion_modelo) + 1)
            )

            resultados_validacion_todos.append(
                df_validacion_modelo
            )

            reportes_clasificacion[nombre_modelo] = (
                crear_reporte_clasificacion(
                    y_real=Y_test,
                    y_predicho=Y_test_pred,
                    nombre_modelo=nombre_modelo
                )
            )

            matrices_confusion[nombre_modelo] = (
                crear_tabla_matrices_confusion(
                    y_real=Y_test,
                    y_predicho=Y_test_pred,
                    nombre_modelo=nombre_modelo
                )
            )

            df_predicciones = (
                df_modelado
                .iloc[indices_test][
                    [
                        "idx_feature_matrix",
                        "ResponseId",
                        "pregunta_origen",
                        "comentario_original"
                    ]
                ]
                .reset_index(drop=True)
                .copy()
            )

            for indice_categoria, categoria in enumerate(CAT_COLS):
                df_predicciones[f"{categoria}_real"] = (
                    Y_test[:, indice_categoria]
                )

                df_predicciones[f"{categoria}_predicha"] = (
                    Y_test_pred[:, indice_categoria]
                )

            predicciones_test_modelos[nombre_modelo] = (
                df_predicciones
            )

            modelos_finales[nombre_modelo] = {
                "representacion": representacion,
                "nombre_version": nombre_version,
                "columna_texto": columna_texto,
                "columna_tokens": columna_tokens,
                "parametros": parametros_ganadores,
                "objeto_representacion": objeto_representacion_final,
                "modelo_knn": modelo_final_knn
            }

            tiempo_total_experimento = (
                time.perf_counter() - inicio_experimento
            )

            resultados_finales.append({
                "nombre_modelo": nombre_modelo,
                "representacion": representacion,
                "version_texto": nombre_version,
                "columna_texto": columna_texto,
                "columna_tokens": columna_tokens,
                "n_neighbors": parametros_ganadores["n_neighbors"],
                "weights": parametros_ganadores["weights"],
                "metric": parametros_ganadores["metric"],
                "p": parametros_ganadores["p"],
                "algorithm": parametros_ganadores["algorithm"],
                "f1_macro_validacion": mejor_configuracion["f1_macro"],
                "f1_micro_validacion": mejor_configuracion["f1_micro"],
                "tiempo_representacion_validacion_seg": (
                    metadatos_validacion["tiempo_representacion_seg"]
                ),
                "tiempo_busqueda_hiperparametros_seg": (
                    tiempo_busqueda_hiperparametros
                ),
                "tiempo_representacion_final_seg": (
                    metadatos_finales_representacion[
                        "tiempo_representacion_seg"
                    ]
                ),
                "tiempo_entrenamiento_final_knn_seg": (
                    tiempo_entrenamiento_final
                ),
                "tiempo_clasificacion_test_seg": (
                    tiempo_clasificacion_test
                ),
                "tiempo_clasificacion_test_ms_por_comentario": (
                    tiempo_clasificacion_test / len(Y_test)
                ) * 1000,
                "tiempo_total_experimento_seg": (
                    tiempo_total_experimento
                ),
                "n_features_final": (
                    metadatos_finales_representacion["n_features"]
                ),
                "n_comentarios_train": len(Y_train),
                "n_comentarios_validation": len(Y_val),
                "n_comentarios_test": len(Y_test),
                **metricas_test
            })

            print("Modelo finalizado correctamente.")
            print(
                f"Mejor F1 macro validación: "
                f"{mejor_configuracion['f1_macro']:.4f}"
            )
            print(
                f"F1 macro test: "
                f"{metricas_test['f1_macro']:.4f}"
            )
            print(
                f"Tiempo de clasificación test: "
                f"{tiempo_clasificacion_test:.4f} segundos"
            )

        except Exception as error:

            errores_experimentos.append({
                "nombre_modelo": nombre_modelo,
                "representacion": representacion,
                "version_texto": nombre_version,
                "error": repr(error)
            })

            print("ERROR EN EL EXPERIMENTO:")
            print(repr(error))

if len(resultados_finales) == 0:
    raise RuntimeError(
        "No se completó ningún experimento KNN. "
        "Revisa la tabla errores_experimentos."
    )

df_resultados_finales = pd.DataFrame(resultados_finales)

df_resultados_validacion = pd.concat(
    resultados_validacion_todos,
    ignore_index=True
)

df_errores_experimentos = pd.DataFrame(
    errores_experimentos
)

print("\n" + "=" * 85)
print("EJECUCIÓN DE EXPERIMENTOS FINALIZADA")
print("=" * 85)
print(f"Modelos finalizados correctamente: {len(df_resultados_finales)}")
print(f"Experimentos con error: {len(df_errores_experimentos)}")

if not df_errores_experimentos.empty:
    display(df_errores_experimentos)


EXPERIMENTO 1/10: KNN__TFIDF_NGRAMS__texto_v1_limpio
Modelo finalizado correctamente.
Mejor F1 macro validación: 0.7595
F1 macro test: 0.7900
Tiempo de clasificación test: 0.0249 segundos

EXPERIMENTO 2/10: KNN__WORD2VEC__texto_v1_limpio
Modelo finalizado correctamente.
Mejor F1 macro validación: 0.6395
F1 macro test: 0.6410
Tiempo de clasificación test: 0.0596 segundos

EXPERIMENTO 3/10: KNN__TFIDF_NGRAMS__texto_v2_sin_tildes
Modelo finalizado correctamente.
Mejor F1 macro validación: 0.7708
F1 macro test: 0.7935
Tiempo de clasificación test: 0.0310 segundos

EXPERIMENTO 4/10: KNN__WORD2VEC__texto_v2_sin_tildes
Modelo finalizado correctamente.
Mejor F1 macro validación: 0.6598
F1 macro test: 0.6242
Tiempo de clasificación test: 0.0277 segundos

EXPERIMENTO 5/10: KNN__TFIDF_NGRAMS__texto_v3_sin_stopwords
Modelo finalizado correctamente.
Mejor F1 macro validación: 0.7796
F1 macro test: 0.7983
Tiempo de clasificación test: 0.0533 segundos

EXPERIMENTO 6/10: KNN__WORD2VEC__texto_v3_sin_s

### Etapa KNN 5: Ranking comparativo y mejor configuración base

Se consolidan las métricas de los diez modelos, se ordenan según desempeño y se identifica la configuración base con mejor resultado antes de calibrar umbrales.

In [37]:
df_resultados_finales = (
    df_resultados_finales
    .sort_values(
        by=[
            "f1_macro",
            "f1_micro",
            "precision_macro",
            "tiempo_clasificacion_test_seg"
        ],
        ascending=[
            False,
            False,
            False,
            True
        ]
    )
    .reset_index(drop=True)
)

df_resultados_finales.insert(
    0,
    "ranking_final",
    np.arange(1, len(df_resultados_finales) + 1)
)

nombre_mejor_modelo = df_resultados_finales.loc[
    0,
    "nombre_modelo"
]

df_mejor_modelo = df_resultados_finales.head(1).copy()

columnas_resumen = [
    "ranking_final",
    "nombre_modelo",
    "representacion",
    "version_texto",
    "n_neighbors",
    "weights",
    "metric",
    "p",
    "algorithm",
    "accuracy_exacta",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "precision_micro",
    "recall_micro",
    "f1_micro",
    "f1_weighted",
    "hamming_loss",
    "tiempo_representacion_final_seg",
    "tiempo_entrenamiento_final_knn_seg",
    "tiempo_clasificacion_test_seg",
    "tiempo_clasificacion_test_ms_por_comentario",
    "tiempo_total_experimento_seg"
]

print("=" * 85)
print("RANKING FINAL DE LOS MODELOS KNN")
print("=" * 85)

display(
    df_resultados_finales[
        columnas_resumen
    ].round(4)
)

print("\nMEJOR CONFIGURACIÓN FINAL:")
display(
    df_mejor_modelo[
        columnas_resumen
    ].round(4)
)

print(
    "\nNota metodológica: en KNN el tiempo de entrenamiento suele ser "
    "muy bajo porque el algoritmo almacena los ejemplos. Por eso el "
    "tiempo de clasificación/predicción es una métrica relevante."
)

df_top_validacion_por_modelo = (
    df_resultados_validacion
    .sort_values(
        by=[
            "nombre_modelo",
            "f1_macro",
            "f1_micro",
            "tiempo_prediccion_validacion_seg"
        ],
        ascending=[
            True,
            False,
            False,
            True
        ]
    )
    .groupby("nombre_modelo", as_index=False)
    .head(3)
    .sort_values(
        by=[
            "f1_macro",
            "f1_micro"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nTOP 3 CONFIGURACIONES DE VALIDACIÓN POR EXPERIMENTO:")
display(
    df_top_validacion_por_modelo[
        [
            "nombre_modelo",
            "ranking_validacion",
            "n_neighbors",
            "weights",
            "metric",
            "p",
            "algorithm",
            "f1_macro",
            "f1_micro",
            "tiempo_prediccion_validacion_seg"
        ]
    ].round(4)
)

RANKING FINAL DE LOS MODELOS KNN


,ranking_final,nombre_modelo,representacion,version_texto,n_neighbors,weights,metric,p,algorithm,accuracy_exacta,precision_macro,recall_macro,f1_macro,precision_micro,recall_micro,f1_micro,f1_weighted,hamming_loss,tiempo_representacion_final_seg,tiempo_entrenamiento_final_knn_seg,tiempo_clasificacion_test_seg,tiempo_clasificacion_test_ms_por_comentario,tiempo_total_experimento_seg
0,1,KNN__TFIDF_NGRAMS__texto_v5_lemma,TFIDF_NGRAMS,texto_v5_lemma,5,distance,minkowski,2.0,brute,0.7265,0.9117,0.7377,0.8116,0.9137,0.7377,0.8163,0.8126,0.0726,0.0343,0.0005,0.0269,0.1208,1.2647
1,2,KNN__TFIDF_NGRAMS__texto_v4_stemming,TFIDF_NGRAMS,texto_v4_stemming,3,uniform,cosine,NaN,brute,0.7354,0.8713,0.7543,0.8051,0.8638,0.7541,0.8053,0.8064,0.0798,0.0223,0.0007,0.0582,0.2611,1.2762
2,3,KNN__TFIDF_NGRAMS__texto_v3_sin_stopwords,TFIDF_NGRAMS,texto_v3_sin_stopwords,3,uniform,cosine,NaN,brute,0.7175,0.8732,0.7469,0.7983,0.8585,0.7459,0.7982,0.8000,0.0825,0.0220,0.0007,0.0533,0.2389,1.2456
3,4,KNN__TFIDF_NGRAMS__texto_v2_sin_tildes,TFIDF_NGRAMS,texto_v2_sin_tildes,7,distance,cosine,NaN,brute,0.6906,0.9183,0.7051,0.7935,0.9149,0.7049,0.7963,0.7944,0.0789,0.0284,0.0007,0.0310,0.1391,1.4430
4,5,KNN__TFIDF_NGRAMS__texto_v1_limpio,TFIDF_NGRAMS,texto_v1_limpio,7,distance,cosine,NaN,brute,0.6951,0.9132,0.7051,0.7900,0.9053,0.7049,0.7926,0.7911,0.0807,0.0346,0.0012,0.0249,0.1115,5.0178
5,6,KNN__WORD2VEC__texto_v4_stemming,WORD2VEC,texto_v4_stemming,3,distance,cosine,NaN,brute,0.7354,0.8299,0.7498,0.7825,0.8243,0.7500,0.7854,0.7821,0.0897,0.1175,0.0007,0.0321,0.1437,3.9046
6,7,KNN__WORD2VEC__texto_v3_sin_stopwords,WORD2VEC,texto_v3_sin_stopwords,5,distance,cosine,NaN,brute,0.5830,0.7790,0.6071,0.6813,0.7749,0.6066,0.6805,0.6804,0.1247,0.1266,0.0008,0.0273,0.1223,3.9238
7,8,KNN__WORD2VEC__texto_v5_lemma,WORD2VEC,texto_v5_lemma,3,distance,cosine,NaN,brute,0.5874,0.7380,0.6272,0.6702,0.7183,0.6270,0.6696,0.6696,0.1354,0.1620,0.0010,0.0191,0.0856,4.0107
8,9,KNN__WORD2VEC__texto_v1_limpio,WORD2VEC,texto_v1_limpio,7,uniform,cosine,NaN,brute,0.5471,0.8383,0.5489,0.6410,0.7976,0.5492,0.6505,0.6413,0.1291,0.1573,0.0008,0.0596,0.2673,4.1239
9,10,KNN__WORD2VEC__texto_v2_sin_tildes,WORD2VEC,texto_v2_sin_tildes,3,distance,cosine,NaN,brute,0.5471,0.7005,0.5859,0.6242,0.6682,0.5861,0.6245,0.6237,0.1543,0.1582,0.0007,0.0277,0.1241,3.9696



MEJOR CONFIGURACIÓN FINAL:


,ranking_final,nombre_modelo,representacion,version_texto,n_neighbors,weights,metric,p,algorithm,accuracy_exacta,precision_macro,recall_macro,f1_macro,precision_micro,recall_micro,f1_micro,f1_weighted,hamming_loss,tiempo_representacion_final_seg,tiempo_entrenamiento_final_knn_seg,tiempo_clasificacion_test_seg,tiempo_clasificacion_test_ms_por_comentario,tiempo_total_experimento_seg
0,1,KNN__TFIDF_NGRAMS__texto_v5_lemma,TFIDF_NGRAMS,texto_v5_lemma,5,distance,minkowski,2.0,brute,0.7265,0.9117,0.7377,0.8116,0.9137,0.7377,0.8163,0.8126,0.0726,0.0343,0.0005,0.0269,0.1208,1.2647



Nota metodológica: en KNN el tiempo de entrenamiento suele ser muy bajo porque el algoritmo almacena los ejemplos. Por eso el tiempo de clasificación/predicción es una métrica relevante.

TOP 3 CONFIGURACIONES DE VALIDACIÓN POR EXPERIMENTO:


,nombre_modelo,ranking_validacion,n_neighbors,weights,metric,p,algorithm,f1_macro,f1_micro,tiempo_prediccion_validacion_seg
0,KNN__TFIDF_NGRAMS__texto_v4_stemming,1,3,uniform,cosine,NaN,brute,0.8220,0.8225,0.0623
1,KNN__TFIDF_NGRAMS__texto_v4_stemming,2,7,distance,cosine,NaN,brute,0.8192,0.8194,0.0209
2,KNN__TFIDF_NGRAMS__texto_v4_stemming,3,9,distance,cosine,NaN,brute,0.8175,0.8176,0.0183
3,KNN__TFIDF_NGRAMS__texto_v5_lemma,1,5,distance,minkowski,2.0,brute,0.7884,0.7910,0.0277
4,KNN__TFIDF_NGRAMS__texto_v5_lemma,2,5,distance,cosine,NaN,brute,0.7872,0.7892,0.0341
5,KNN__TFIDF_NGRAMS__texto_v5_lemma,3,5,uniform,minkowski,2.0,brute,0.7853,0.7883,0.0272
6,KNN__TFIDF_NGRAMS__texto_v3_sin_stopwords,1,3,uniform,cosine,NaN,brute,0.7796,0.7809,0.0541
7,KNN__WORD2VEC__texto_v4_stemming,1,3,distance,cosine,NaN,brute,0.7779,0.7787,0.0185
8,KNN__TFIDF_NGRAMS__texto_v3_sin_stopwords,2,3,distance,cosine,NaN,brute,0.7778,0.7792,0.0204
9,KNN__TFIDF_NGRAMS__texto_v3_sin_stopwords,3,5,distance,cosine,NaN,brute,0.7774,0.7788,0.0194


### Etapa KNN 6: Reportes de clasificación y matrices de confusión

Se generan reportes por categoría y matrices de confusión multietiqueta para revisar el comportamiento de los modelos y las clases más difíciles.

In [38]:
df_reportes_consolidados = pd.concat(
    reportes_clasificacion.values(),
    ignore_index=True
)

df_matrices_confusion_consolidadas = pd.concat(
    matrices_confusion.values(),
    ignore_index=True
)

print("=" * 85)
print("CLASSIFICATION REPORT CONSOLIDADO")
print("=" * 85)

display(
    df_reportes_consolidados.round(4)
)

print("=" * 85)
print("MATRICES DE CONFUSIÓN CONSOLIDADAS")
print("=" * 85)

display(
    df_matrices_confusion_consolidadas
)

MODELO_A_REVISAR = nombre_mejor_modelo

print("=" * 85)
print(f"DETALLE DEL MODELO: {MODELO_A_REVISAR}")
print("=" * 85)

print("\nClassification report:")
display(
    reportes_clasificacion[MODELO_A_REVISAR].round(4)
)

print("\nMatriz de confusión por categoría:")

df_confusion_modelo = matrices_confusion[
    MODELO_A_REVISAR
].copy()

for categoria in CAT_COLS:

    fila = df_confusion_modelo.loc[
        df_confusion_modelo["categoria"].eq(categoria)
    ].iloc[0]

    matriz = pd.DataFrame(
        [
            [fila["TN"], fila["FP"]],
            [fila["FN"], fila["TP"]]
        ],
        index=[
            "Real 0",
            "Real 1"
        ],
        columns=[
            "Predicho 0",
            "Predicho 1"
        ]
    )

    print(f"\nCategoría: {categoria}")
    display(matriz)

CLASSIFICATION REPORT CONSOLIDADO


,nombre_modelo,clase,precision,recall,f1-score,support
0,KNN__TFIDF_NGRAMS__texto_v1_limpio,cat_Pago_monto_y_calculo,0.9535,0.8039,0.8723,51.0
1,KNN__TFIDF_NGRAMS__texto_v1_limpio,cat_Tiempos_y_oportunidad,0.9231,0.7059,0.8000,51.0
2,KNN__TFIDF_NGRAMS__texto_v1_limpio,cat_Informacion_y_comunicacion,1.0000,0.5918,0.7436,49.0
3,KNN__TFIDF_NGRAMS__texto_v1_limpio,cat_Atencion_y_gestion_del_servicio,0.9118,0.6458,0.7561,48.0
4,KNN__TFIDF_NGRAMS__texto_v1_limpio,cat_Satisfaccion_o_comentario_general,0.7778,0.7778,0.7778,45.0
...,...,...,...,...,...,...
85,KNN__WORD2VEC__texto_v5_lemma,cat_Satisfaccion_o_comentario_general,0.9032,0.6222,0.7368,45.0
86,KNN__WORD2VEC__texto_v5_lemma,micro avg,0.7183,0.6270,0.6696,244.0
87,KNN__WORD2VEC__texto_v5_lemma,macro avg,0.7380,0.6272,0.6702,244.0
88,KNN__WORD2VEC__texto_v5_lemma,weighted avg,0.7366,0.6270,0.6696,244.0


MATRICES DE CONFUSIÓN CONSOLIDADAS


,nombre_modelo,categoria,TN,FP,FN,TP
0,KNN__TFIDF_NGRAMS__texto_v1_limpio,cat_Pago_monto_y_calculo,170,2,10,41
1,KNN__TFIDF_NGRAMS__texto_v1_limpio,cat_Tiempos_y_oportunidad,169,3,15,36
2,KNN__TFIDF_NGRAMS__texto_v1_limpio,cat_Informacion_y_comunicacion,174,0,20,29
3,KNN__TFIDF_NGRAMS__texto_v1_limpio,cat_Atencion_y_gestion_del_servicio,172,3,17,31
4,KNN__TFIDF_NGRAMS__texto_v1_limpio,cat_Satisfaccion_o_comentario_general,168,10,10,35
5,KNN__WORD2VEC__texto_v1_limpio,cat_Pago_monto_y_calculo,160,12,20,31
6,KNN__WORD2VEC__texto_v1_limpio,cat_Tiempos_y_oportunidad,171,1,23,28
7,KNN__WORD2VEC__texto_v1_limpio,cat_Informacion_y_comunicacion,173,1,33,16
8,KNN__WORD2VEC__texto_v1_limpio,cat_Atencion_y_gestion_del_servicio,158,17,12,36
9,KNN__WORD2VEC__texto_v1_limpio,cat_Satisfaccion_o_comentario_general,175,3,22,23


DETALLE DEL MODELO: KNN__TFIDF_NGRAMS__texto_v5_lemma

Classification report:


...

,Predicho 0,Predicho 1
Real 0,170,4
Real 1,23,26



Categoría: cat_Atencion_y_gestion_del_servicio


,Predicho 0,Predicho 1
Real 0,171,4
Real 1,12,36



Categoría: cat_Satisfaccion_o_comentario_general


,Predicho 0,Predicho 1
Real 0,171,7
Real 1,9,36


### Etapa KNN 7: Ajuste de umbrales por categoría

Se calibran umbrales de decisión independientes para las categorías de cada modelo. La selección se realiza solo con validación y se compara la predicción estándar con la calibrada sobre el conjunto de prueba.

In [39]:
import time
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    hamming_loss,
    classification_report,
    multilabel_confusion_matrix
)

BETA_UMBRAL = 2

UMBRALES_CANDIDATOS = np.round(
    np.arange(0.05, 0.96, 0.05),
    2
)

UMBRAL_ESTANDAR = 0.50

objetos_requeridos_umbral = [
    "df_modelado",
    "VERSIONES_EXPERIMENTO",
    "CAT_COLS",
    "df_resultados_finales",
    "indices_train",
    "indices_val",
    "indices_train_val",
    "indices_test",
    "Y_train",
    "Y_val",
    "Y_train_val",
    "Y_test",
    "texto_seguro",
    "construir_tfidf",
    "construir_word2vec",
    "crear_knn",
    "calcular_metricas_multietiqueta"
]

faltantes_umbral = [
    objeto
    for objeto in objetos_requeridos_umbral
    if objeto not in globals()
]

if faltantes_umbral:
    raise RuntimeError(
        "Faltan variables o funciones necesarias. "
        "Ejecuta primero las etapas KNN 1 a 6.\n\n"
        f"Faltantes: {faltantes_umbral}"
    )

def obtener_probabilidades_knn_multietiqueta(modelo_knn, X):
    """Obtiene probabilidades positivas por categoría."""

    salida_proba = modelo_knn.predict_proba(X)

    if not isinstance(salida_proba, list):
        raise TypeError(
            "Se esperaba una lista de probabilidades por categoría. "
            "Revisa que KNN haya sido entrenado con múltiples columnas cat_."
        )

    if len(salida_proba) != len(CAT_COLS):
        raise ValueError(
            "La cantidad de salidas de predict_proba no coincide con CAT_COLS."
        )

    probabilidades = np.zeros(
        (X.shape[0], len(CAT_COLS)),
        dtype=np.float32
    )

    clases_por_categoria = modelo_knn.classes_

    for indice_categoria, proba_categoria in enumerate(salida_proba):

        clases_categoria = np.asarray(
            clases_por_categoria[indice_categoria]
        )

        if 1 in clases_categoria:
            indice_clase_positiva = np.where(
                clases_categoria == 1
            )[0][0]

            probabilidades[:, indice_categoria] = (
                proba_categoria[:, indice_clase_positiva]
            )

        else:

            clase_unica = int(clases_categoria[0])

            probabilidades[:, indice_categoria] = (
                1.0 if clase_unica == 1 else 0.0
            )

    return probabilidades

def convertir_probabilidades_a_etiquetas(
    probabilidades,
    umbrales_por_categoria
):
    """Convierte probabilidades en etiquetas binarias."""

    umbrales_array = np.asarray(
        umbrales_por_categoria,
        dtype=float
    ).reshape(1, -1)

    return (probabilidades >= umbrales_array).astype(int)

def calcular_metricas_con_f2(y_real, y_predicho):
    """Calcula métricas multietiqueta y F2."""

    metricas = calcular_metricas_multietiqueta(
        y_real,
        y_predicho
    )

    metricas["f2_macro"] = fbeta_score(
        y_real,
        y_predicho,
        beta=BETA_UMBRAL,
        average="macro",
        zero_division=0
    )

    metricas["f2_micro"] = fbeta_score(
        y_real,
        y_predicho,
        beta=BETA_UMBRAL,
        average="micro",
        zero_division=0
    )

    return metricas

def buscar_umbrales_por_categoria(
    y_validation_real,
    probabilidades_validation
):
    """Selecciona umbrales por categoría en validación."""

    resultados_busqueda = []

    for indice_categoria, categoria in enumerate(CAT_COLS):

        y_real_categoria = y_validation_real[:, indice_categoria]
        proba_categoria = probabilidades_validation[:, indice_categoria]

        filas_categoria = []

        for umbral in UMBRALES_CANDIDATOS:

            y_pred_categoria = (
                proba_categoria >= umbral
            ).astype(int)

            precision = precision_score(
                y_real_categoria,
                y_pred_categoria,
                zero_division=0
            )

            recall = recall_score(
                y_real_categoria,
                y_pred_categoria,
                zero_division=0
            )

            f1 = f1_score(
                y_real_categoria,
                y_pred_categoria,
                zero_division=0
            )

            f2 = fbeta_score(
                y_real_categoria,
                y_pred_categoria,
                beta=BETA_UMBRAL,
                zero_division=0
            )

            filas_categoria.append({
                "categoria": categoria,
                "umbral": umbral,
                "precision_validation": precision,
                "recall_validation": recall,
                "f1_validation": f1,
                "f2_validation": f2,
                "positivos_reales_validation": int(
                    y_real_categoria.sum()
                ),
                "positivos_predichos_validation": int(
                    y_pred_categoria.sum()
                ),
                "distancia_umbral_estandar": abs(
                    umbral - UMBRAL_ESTANDAR
                )
            })

        df_categoria = pd.DataFrame(filas_categoria)

        mejor_indice = (
            df_categoria
            .sort_values(
                by=[
                    "f2_validation",
                    "recall_validation",
                    "f1_validation",
                    "precision_validation",
                    "distancia_umbral_estandar"
                ],
                ascending=[
                    False,
                    False,
                    False,
                    False,
                    True
                ]
            )
            .index[0]
        )

        df_categoria["seleccionado"] = False
        df_categoria.loc[mejor_indice, "seleccionado"] = True

        resultados_busqueda.append(df_categoria)

    df_busqueda = pd.concat(
        resultados_busqueda,
        ignore_index=True
    )

    df_umbrales_seleccionados = (
        df_busqueda
        .loc[df_busqueda["seleccionado"]]
        .sort_values("categoria")
        .reset_index(drop=True)
    )

    df_umbrales_seleccionados = (
        df_umbrales_seleccionados
        .set_index("categoria")
        .reindex(CAT_COLS)
        .reset_index()
    )

    umbrales = df_umbrales_seleccionados[
        "umbral"
    ].to_numpy(dtype=float)

    return (
        df_busqueda,
        df_umbrales_seleccionados,
        umbrales
    )

def crear_reporte_umbral(y_real, y_predicho, nombre_modelo):
    """Genera el reporte con umbrales calibrados."""

    reporte = classification_report(
        y_real,
        y_predicho,
        target_names=CAT_COLS,
        output_dict=True,
        zero_division=0
    )

    df_reporte = (
        pd.DataFrame(reporte)
        .T
        .reset_index()
        .rename(columns={"index": "clase"})
    )

    df_reporte.insert(
        0,
        "nombre_modelo",
        nombre_modelo
    )

    return df_reporte

def crear_matriz_confusion_umbral(
    y_real,
    y_predicho,
    nombre_modelo
):
    """Genera matrices de confusión con umbrales calibrados."""

    matrices = multilabel_confusion_matrix(
        y_real,
        y_predicho
    )

    filas = []

    for indice_categoria, categoria in enumerate(CAT_COLS):

        tn, fp, fn, tp = matrices[
            indice_categoria
        ].ravel()

        filas.append({
            "nombre_modelo": nombre_modelo,
            "categoria": categoria,
            "TN": int(tn),
            "FP": int(fp),
            "FN": int(fn),
            "TP": int(tp)
        })

    return pd.DataFrame(filas)

resultados_umbral_modelos = []
resultados_busqueda_umbrales = []
umbrales_seleccionados_todos = []

reportes_umbrales_test = {}
matrices_confusion_umbrales_test = {}
predicciones_umbrales_test = {}
modelos_umbrales_finales = {}

errores_umbral = []

total_modelos_umbral = len(VERSIONES_EXPERIMENTO) * 2
contador_modelo_umbral = 0

for configuracion in VERSIONES_EXPERIMENTO:

    nombre_version = configuracion["nombre_version"]
    columna_texto = configuracion["columna_texto"]
    columna_tokens = configuracion["columna_tokens"]

    for representacion in ["TFIDF_NGRAMS", "WORD2VEC"]:

        contador_modelo_umbral += 1

        nombre_modelo = (
            f"KNN__{representacion}__{nombre_version}"
        )

        print("\n" + "=" * 90)
        print(
            f"AJUSTE DE UMBRALES {contador_modelo_umbral}/"
            f"{total_modelos_umbral}: {nombre_modelo}"
        )
        print("=" * 90)

        try:

            fila_configuracion = df_resultados_finales.loc[
                df_resultados_finales["nombre_modelo"].eq(
                    nombre_modelo
                )
            ]

            if len(fila_configuracion) != 1:
                raise ValueError(
                    "No se encontró una única configuración final "
                    f"para {nombre_modelo}."
                )

            fila_configuracion = fila_configuracion.iloc[0]

            parametros_knn = {
                "n_neighbors": int(
                    fila_configuracion["n_neighbors"]
                ),
                "weights": fila_configuracion["weights"],
                "metric": fila_configuracion["metric"],
                "algorithm": fila_configuracion["algorithm"],
                "p": (
                    None
                    if pd.isna(fila_configuracion["p"])
                    else int(fila_configuracion["p"])
                )
            }

            textos_train = (
                df_modelado
                .iloc[indices_train][columna_texto]
                .apply(texto_seguro)
                .to_numpy()
            )

            textos_val = (
                df_modelado
                .iloc[indices_val][columna_texto]
                .apply(texto_seguro)
                .to_numpy()
            )

            textos_train_val = (
                df_modelado
                .iloc[indices_train_val][columna_texto]
                .apply(texto_seguro)
                .to_numpy()
            )

            textos_test = (
                df_modelado
                .iloc[indices_test][columna_texto]
                .apply(texto_seguro)
                .to_numpy()
            )

            tokens_train = (
                df_modelado
                .iloc[indices_train][columna_tokens]
                .tolist()
            )

            tokens_val = (
                df_modelado
                .iloc[indices_val][columna_tokens]
                .tolist()
            )

            tokens_train_val = (
                df_modelado
                .iloc[indices_train_val][columna_tokens]
                .tolist()
            )

            tokens_test = (
                df_modelado
                .iloc[indices_test][columna_tokens]
                .tolist()
            )

            if representacion == "TFIDF_NGRAMS":

                (
                    _,
                    X_train_validacion,
                    X_val_actual,
                    metadatos_validacion_umbral
                ) = construir_tfidf(
                    textos_ajuste=textos_train,
                    textos_evaluacion=textos_val
                )

            else:

                (
                    _,
                    X_train_validacion,
                    X_val_actual,
                    metadatos_validacion_umbral
                ) = construir_word2vec(
                    tokens_ajuste=tokens_train,
                    tokens_evaluacion=tokens_val
                )

            modelo_validacion = crear_knn(
                parametros_knn
            )

            modelo_validacion.fit(
                X_train_validacion,
                Y_train
            )

            Y_val_pred_base = modelo_validacion.predict(
                X_val_actual
            ).astype(int)

            probabilidades_val = (
                obtener_probabilidades_knn_multietiqueta(
                    modelo_validacion,
                    X_val_actual
                )
            )

            (
                df_busqueda_modelo,
                df_umbrales_modelo,
                umbrales_modelo
            ) = buscar_umbrales_por_categoria(
                y_validation_real=Y_val,
                probabilidades_validation=probabilidades_val
            )

            Y_val_pred_umbral = (
                convertir_probabilidades_a_etiquetas(
                    probabilidades_val,
                    umbrales_modelo
                )
            )

            metricas_val_base = calcular_metricas_con_f2(
                Y_val,
                Y_val_pred_base
            )

            metricas_val_umbral = calcular_metricas_con_f2(
                Y_val,
                Y_val_pred_umbral
            )

            df_busqueda_modelo.insert(
                0,
                "nombre_modelo",
                nombre_modelo
            )

            df_busqueda_modelo.insert(
                1,
                "representacion",
                representacion
            )

            df_busqueda_modelo.insert(
                2,
                "version_texto",
                nombre_version
            )

            df_umbrales_modelo.insert(
                0,
                "nombre_modelo",
                nombre_modelo
            )

            df_umbrales_modelo.insert(
                1,
                "representacion",
                representacion
            )

            df_umbrales_modelo.insert(
                2,
                "version_texto",
                nombre_version
            )

            resultados_busqueda_umbrales.append(
                df_busqueda_modelo
            )

            umbrales_seleccionados_todos.append(
                df_umbrales_modelo
            )

            if representacion == "TFIDF_NGRAMS":

                (
                    objeto_representacion_final,
                    X_train_val_actual,
                    X_test_actual,
                    metadatos_test_umbral
                ) = construir_tfidf(
                    textos_ajuste=textos_train_val,
                    textos_evaluacion=textos_test
                )

            else:

                (
                    objeto_representacion_final,
                    X_train_val_actual,
                    X_test_actual,
                    metadatos_test_umbral
                ) = construir_word2vec(
                    tokens_ajuste=tokens_train_val,
                    tokens_evaluacion=tokens_test
                )

            modelo_final_umbral = crear_knn(
                parametros_knn
            )

            inicio_fit_final = time.perf_counter()

            modelo_final_umbral.fit(
                X_train_val_actual,
                Y_train_val
            )

            tiempo_fit_final = (
                time.perf_counter() - inicio_fit_final
            )

            inicio_prediccion_base = time.perf_counter()

            Y_test_pred_base = modelo_final_umbral.predict(
                X_test_actual
            ).astype(int)

            tiempo_prediccion_base = (
                time.perf_counter() - inicio_prediccion_base
            )

            inicio_probabilidades_test = time.perf_counter()

            probabilidades_test = (
                obtener_probabilidades_knn_multietiqueta(
                    modelo_final_umbral,
                    X_test_actual
                )
            )

            Y_test_pred_umbral = (
                convertir_probabilidades_a_etiquetas(
                    probabilidades_test,
                    umbrales_modelo
                )
            )

            tiempo_probabilidades_umbral = (
                time.perf_counter() - inicio_probabilidades_test
            )

            metricas_test_base = calcular_metricas_con_f2(
                Y_test,
                Y_test_pred_base
            )

            metricas_test_umbral = calcular_metricas_con_f2(
                Y_test,
                Y_test_pred_umbral
            )

            reportes_umbrales_test[nombre_modelo] = (
                crear_reporte_umbral(
                    Y_test,
                    Y_test_pred_umbral,
                    nombre_modelo
                )
            )

            matrices_confusion_umbrales_test[nombre_modelo] = (
                crear_matriz_confusion_umbral(
                    Y_test,
                    Y_test_pred_umbral,
                    nombre_modelo
                )
            )

            predicciones_umbrales_test[nombre_modelo] = {
                "probabilidades_test": probabilidades_test,
                "prediccion_base": Y_test_pred_base,
                "prediccion_umbral": Y_test_pred_umbral,
                "umbrales_por_categoria": dict(
                    zip(CAT_COLS, umbrales_modelo)
                )
            }

            modelos_umbrales_finales[nombre_modelo] = {
                "representacion": representacion,
                "version_texto": nombre_version,
                "columna_texto": columna_texto,
                "columna_tokens": columna_tokens,
                "parametros_knn": parametros_knn,
                "objeto_representacion": objeto_representacion_final,
                "modelo_knn": modelo_final_umbral,
                "umbrales_por_categoria": dict(
                    zip(CAT_COLS, umbrales_modelo)
                )
            }

            resultados_umbral_modelos.append({
                "nombre_modelo": nombre_modelo,
                "representacion": representacion,
                "version_texto": nombre_version,
                "n_neighbors": parametros_knn["n_neighbors"],
                "weights": parametros_knn["weights"],
                "metric": parametros_knn["metric"],
                "p": parametros_knn["p"],
                "algorithm": parametros_knn["algorithm"],

                "f1_macro_validacion_base": (
                    metricas_val_base["f1_macro"]
                ),
                "f1_macro_validacion_umbral": (
                    metricas_val_umbral["f1_macro"]
                ),
                "recall_macro_validacion_base": (
                    metricas_val_base["recall_macro"]
                ),
                "recall_macro_validacion_umbral": (
                    metricas_val_umbral["recall_macro"]
                ),
                "f2_macro_validacion_base": (
                    metricas_val_base["f2_macro"]
                ),
                "f2_macro_validacion_umbral": (
                    metricas_val_umbral["f2_macro"]
                ),

                "accuracy_exacta_test_base": (
                    metricas_test_base["accuracy_exacta"]
                ),
                "accuracy_exacta_test_umbral": (
                    metricas_test_umbral["accuracy_exacta"]
                ),
                "precision_macro_test_base": (
                    metricas_test_base["precision_macro"]
                ),
                "precision_macro_test_umbral": (
                    metricas_test_umbral["precision_macro"]
                ),
                "recall_macro_test_base": (
                    metricas_test_base["recall_macro"]
                ),
                "recall_macro_test_umbral": (
                    metricas_test_umbral["recall_macro"]
                ),
                "f1_macro_test_base": (
                    metricas_test_base["f1_macro"]
                ),
                "f1_macro_test_umbral": (
                    metricas_test_umbral["f1_macro"]
                ),
                "f2_macro_test_base": (
                    metricas_test_base["f2_macro"]
                ),
                "f2_macro_test_umbral": (
                    metricas_test_umbral["f2_macro"]
                ),
                "f1_micro_test_base": (
                    metricas_test_base["f1_micro"]
                ),
                "f1_micro_test_umbral": (
                    metricas_test_umbral["f1_micro"]
                ),
                "hamming_loss_test_base": (
                    metricas_test_base["hamming_loss"]
                ),
                "hamming_loss_test_umbral": (
                    metricas_test_umbral["hamming_loss"]
                ),

                "delta_recall_macro_test": (
                    metricas_test_umbral["recall_macro"]
                    - metricas_test_base["recall_macro"]
                ),
                "delta_precision_macro_test": (
                    metricas_test_umbral["precision_macro"]
                    - metricas_test_base["precision_macro"]
                ),
                "delta_f1_macro_test": (
                    metricas_test_umbral["f1_macro"]
                    - metricas_test_base["f1_macro"]
                ),
                "delta_f2_macro_test": (
                    metricas_test_umbral["f2_macro"]
                    - metricas_test_base["f2_macro"]
                ),

                "tiempo_representacion_test_seg": (
                    metadatos_test_umbral[
                        "tiempo_representacion_seg"
                    ]
                ),
                "tiempo_entrenamiento_final_knn_seg": (
                    tiempo_fit_final
                ),
                "tiempo_prediccion_estandar_test_seg": (
                    tiempo_prediccion_base
                ),
                "tiempo_prediccion_con_umbrales_test_seg": (
                    tiempo_probabilidades_umbral
                ),
                "tiempo_prediccion_umbral_ms_por_comentario": (
                    tiempo_probabilidades_umbral / len(Y_test)
                ) * 1000
            })

            print("Ajuste finalizado correctamente.")
            print(
                "F2 macro validation: "
                f"{metricas_val_base['f2_macro']:.4f} "
                f"→ {metricas_val_umbral['f2_macro']:.4f}"
            )
            print(
                "Recall macro test: "
                f"{metricas_test_base['recall_macro']:.4f} "
                f"→ {metricas_test_umbral['recall_macro']:.4f}"
            )
            print(
                "F1 macro test: "
                f"{metricas_test_base['f1_macro']:.4f} "
                f"→ {metricas_test_umbral['f1_macro']:.4f}"
            )

        except Exception as error:

            errores_umbral.append({
                "nombre_modelo": nombre_modelo,
                "representacion": representacion,
                "version_texto": nombre_version,
                "error": repr(error)
            })

            print("ERROR EN AJUSTE DE UMBRALES:")
            print(repr(error))

if len(resultados_umbral_modelos) == 0:
    raise RuntimeError(
        "No se logró calibrar ningún modelo. "
        "Revisa df_errores_ajuste_umbrales."
    )

df_resultados_umbrales_test = pd.DataFrame(
    resultados_umbral_modelos
)

df_resultados_umbrales_test = (
    df_resultados_umbrales_test
    .sort_values(
        by=[
            "f2_macro_validacion_umbral",
            "recall_macro_validacion_umbral",
            "f1_macro_validacion_umbral"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

df_resultados_umbrales_test.insert(
    0,
    "ranking_calibrado_validation",
    np.arange(
        1,
        len(df_resultados_umbrales_test) + 1
    )
)

df_busqueda_umbrales_validacion = pd.concat(
    resultados_busqueda_umbrales,
    ignore_index=True
)

df_umbrales_seleccionados_validacion = pd.concat(
    umbrales_seleccionados_todos,
    ignore_index=True
)

df_reportes_umbrales_test = pd.concat(
    reportes_umbrales_test.values(),
    ignore_index=True
)

df_matrices_confusion_umbrales_test = pd.concat(
    matrices_confusion_umbrales_test.values(),
    ignore_index=True
)

df_errores_ajuste_umbrales = pd.DataFrame(
    errores_umbral
)

columnas_resumen_umbrales = [
    "ranking_calibrado_validation",
    "nombre_modelo",
    "representacion",
    "version_texto",
    "f2_macro_validacion_base",
    "f2_macro_validacion_umbral",
    "recall_macro_test_base",
    "recall_macro_test_umbral",
    "delta_recall_macro_test",
    "precision_macro_test_base",
    "precision_macro_test_umbral",
    "delta_precision_macro_test",
    "f1_macro_test_base",
    "f1_macro_test_umbral",
    "delta_f1_macro_test",
    "f2_macro_test_base",
    "f2_macro_test_umbral",
    "delta_f2_macro_test",
    "tiempo_prediccion_umbral_ms_por_comentario"
]

print("\n" + "=" * 95)
print("COMPARACIÓN: PREDICCIÓN BASE VS. UMBRALES CALIBRADOS")
print("=" * 95)

display(
    df_resultados_umbrales_test[
        columnas_resumen_umbrales
    ].round(4)
)

print("\n" + "=" * 95)
print("UMBRALES SELECCIONADOS POR MODELO Y CATEGORÍA")
print("=" * 95)

display(
    df_umbrales_seleccionados_validacion[
        [
            "nombre_modelo",
            "categoria",
            "umbral",
            "precision_validation",
            "recall_validation",
            "f1_validation",
            "f2_validation",
            "positivos_reales_validation",
            "positivos_predichos_validation"
        ]
    ].round(4)
)

print("\n" + "=" * 95)
print("MODELO MEJOR CALIBRADO SEGÚN VALIDATION")
print("=" * 95)

nombre_mejor_modelo_umbral = df_resultados_umbrales_test.loc[
    0,
    "nombre_modelo"
]

print(nombre_mejor_modelo_umbral)

print("\nClassification report del mejor modelo calibrado:")
display(
    reportes_umbrales_test[
        nombre_mejor_modelo_umbral
    ].round(4)
)

print("\nMatriz de confusión por categoría del mejor modelo calibrado:")
display(
    matrices_confusion_umbrales_test[
        nombre_mejor_modelo_umbral
    ]
)

if not df_errores_ajuste_umbrales.empty:
    print("\nExperimentos con error:")
    display(df_errores_ajuste_umbrales)


AJUSTE DE UMBRALES 1/10: KNN__TFIDF_NGRAMS__texto_v1_limpio
Ajuste finalizado correctamente.
F2 macro validation: 0.7147 → 0.8203
Recall macro test: 0.7051 → 0.8343
F1 macro test: 0.7900 → 0.7744

AJUSTE DE UMBRALES 2/10: KNN__WORD2VEC__texto_v1_limpio
Ajuste finalizado correctamente.
F2 macro validation: 0.5882 → 0.7230
Recall macro test: 0.5489 → 0.8995
F1 macro test: 0.6410 → 0.6066

AJUSTE DE UMBRALES 3/10: KNN__TFIDF_NGRAMS__texto_v2_sin_tildes
Ajuste finalizado correctamente.
F2 macro validation: 0.7242 → 0.8269
Recall macro test: 0.7051 → 0.8756
F1 macro test: 0.7935 → 0.7396

AJUSTE DE UMBRALES 4/10: KNN__WORD2VEC__texto_v2_sin_tildes
Ajuste finalizado correctamente.
F2 macro validation: 0.6305 → 0.7423
Recall macro test: 0.5859 → 0.7773
F1 macro test: 0.6242 → 0.6382

AJUSTE DE UMBRALES 5/10: KNN__TFIDF_NGRAMS__texto_v3_sin_stopwords
Ajuste finalizado correctamente.
F2 macro validation: 0.7508 → 0.8214
Recall macro test: 0.7469 → 0.8791
F1 macro test: 0.7983 → 0.7418

AJUSTE 

,ranking_calibrado_validation,nombre_modelo,representacion,version_texto,f2_macro_validacion_base,f2_macro_validacion_umbral,recall_macro_test_base,recall_macro_test_umbral,delta_recall_macro_test,precision_macro_test_base,precision_macro_test_umbral,delta_precision_macro_test,f1_macro_test_base,f1_macro_test_umbral,delta_f1_macro_test,f2_macro_test_base,f2_macro_test_umbral,delta_f2_macro_test,tiempo_prediccion_umbral_ms_por_comentario
0,1,KNN__TFIDF_NGRAMS__texto_v4_stemming,TFIDF_NGRAMS,texto_v4_stemming,0.7926,0.8370,0.7543,0.8600,0.1057,0.8713,0.7493,-0.1220,0.8051,0.7952,-0.0099,0.7731,0.8315,0.0584,0.1390
1,2,KNN__TFIDF_NGRAMS__texto_v5_lemma,TFIDF_NGRAMS,texto_v5_lemma,0.7436,0.8359,0.7377,0.8973,0.1596,0.9117,0.7033,-0.2084,0.8116,0.7767,-0.0349,0.7649,0.8407,0.0758,0.1280
2,3,KNN__TFIDF_NGRAMS__texto_v2_sin_tildes,TFIDF_NGRAMS,texto_v2_sin_tildes,0.7242,0.8269,0.7051,0.8756,0.1705,0.9183,0.6610,-0.2573,0.7935,0.7396,-0.0539,0.7373,0.8111,0.0738,0.1338
3,4,KNN__TFIDF_NGRAMS__texto_v3_sin_stopwords,TFIDF_NGRAMS,texto_v3_sin_stopwords,0.7508,0.8214,0.7469,0.8791,0.1322,0.8732,0.6698,-0.2034,0.7983,0.7418,-0.0565,0.7650,0.8124,0.0474,0.1333
4,5,KNN__TFIDF_NGRAMS__texto_v1_limpio,TFIDF_NGRAMS,texto_v1_limpio,0.7147,0.8203,0.7051,0.8343,0.1292,0.9132,0.7524,-0.1608,0.7900,0.7744,-0.0156,0.7358,0.8036,0.0678,0.1361
5,6,KNN__WORD2VEC__texto_v4_stemming,WORD2VEC,texto_v4_stemming,0.7582,0.8167,0.7498,0.9050,0.1552,0.8299,0.6855,-0.1444,0.7825,0.7773,-0.0053,0.7614,0.8481,0.0867,0.1365
6,7,KNN__WORD2VEC__texto_v3_sin_stopwords,WORD2VEC,texto_v3_sin_stopwords,0.6615,0.7572,0.6071,0.8771,0.2700,0.7790,0.5070,-0.2720,0.6813,0.6393,-0.0420,0.6345,0.7617,0.1271,0.1351
7,8,KNN__WORD2VEC__texto_v5_lemma,WORD2VEC,texto_v5_lemma,0.6578,0.7562,0.6272,0.7898,0.1626,0.7380,0.5256,-0.2124,0.6702,0.6296,-0.0405,0.6419,0.7162,0.0742,0.1332
8,9,KNN__WORD2VEC__texto_v2_sin_tildes,WORD2VEC,texto_v2_sin_tildes,0.6305,0.7423,0.5859,0.7773,0.1914,0.7005,0.5507,-0.1498,0.6242,0.6382,0.0141,0.5977,0.7124,0.1148,0.1350
9,10,KNN__WORD2VEC__texto_v1_limpio,WORD2VEC,texto_v1_limpio,0.5882,0.7230,0.5489,0.8995,0.3506,0.8383,0.4779,-0.3604,0.6410,0.6066,-0.0344,0.5794,0.7445,0.1651,0.0779



UMBRALES SELECCIONADOS POR MODELO Y CATEGORÍA


,nombre_modelo,categoria,umbral,precision_validation,recall_validation,f1_validation,f2_validation,positivos_reales_validation,positivos_predichos_validation
0,KNN__TFIDF_NGRAMS__texto_v1_limpio,cat_Pago_monto_y_calculo,0.20,0.6667,0.9615,0.7874,0.8834,52,75
1,KNN__TFIDF_NGRAMS__texto_v1_limpio,cat_Tiempos_y_oportunidad,0.40,0.7963,0.8431,0.8190,0.8333,51,54
2,KNN__TFIDF_NGRAMS__texto_v1_limpio,cat_Informacion_y_comunicacion,0.40,0.7451,0.7755,0.7600,0.7692,49,51
3,KNN__TFIDF_NGRAMS__texto_v1_limpio,cat_Atencion_y_gestion_del_servicio,0.35,0.8696,0.8333,0.8511,0.8403,48,46
4,KNN__TFIDF_NGRAMS__texto_v1_limpio,cat_Satisfaccion_o_comentario_general,0.15,0.5128,0.8889,0.6504,0.7752,45,78
5,KNN__WORD2VEC__texto_v1_limpio,cat_Pago_monto_y_calculo,0.25,0.5556,0.8654,0.6767,0.7785,52,81
6,KNN__WORD2VEC__texto_v1_limpio,cat_Tiempos_y_oportunidad,0.10,0.3740,0.9608,0.5385,0.7313,51,131
7,KNN__WORD2VEC__texto_v1_limpio,cat_Informacion_y_comunicacion,0.10,0.2885,0.9184,0.4390,0.6392,49,156
8,KNN__WORD2VEC__texto_v1_limpio,cat_Atencion_y_gestion_del_servicio,0.50,0.6452,0.8333,0.7273,0.7874,48,62
9,KNN__WORD2VEC__texto_v1_limpio,cat_Satisfaccion_o_comentario_general,0.10,0.4487,0.7778,0.5691,0.6783,45,78



MODELO MEJOR CALIBRADO SEGÚN VALIDATION
KNN__TFIDF_NGRAMS__texto_v4_stemming

Classification report del mejor modelo calibrado:


,nombre_modelo,clase,precision,recall,f1-score,support
0,KNN__TFIDF_NGRAMS__texto_v4_stemming,cat_Pago_monto_y_calculo,0.7121,0.9216,0.8034,51.0
1,KNN__TFIDF_NGRAMS__texto_v4_stemming,cat_Tiempos_y_oportunidad,0.9070,0.7647,0.8298,51.0
2,KNN__TFIDF_NGRAMS__texto_v4_stemming,cat_Informacion_y_comunicacion,0.7302,0.9388,0.8214,49.0
3,KNN__TFIDF_NGRAMS__texto_v4_stemming,cat_Atencion_y_gestion_del_servicio,0.6774,0.8750,0.7636,48.0
4,KNN__TFIDF_NGRAMS__texto_v4_stemming,cat_Satisfaccion_o_comentario_general,0.7200,0.8000,0.7579,45.0
5,KNN__TFIDF_NGRAMS__texto_v4_stemming,micro avg,0.7394,0.8607,0.7955,244.0
6,KNN__TFIDF_NGRAMS__texto_v4_stemming,macro avg,0.7493,0.8600,0.7952,244.0
7,KNN__TFIDF_NGRAMS__texto_v4_stemming,weighted avg,0.7511,0.8607,0.7963,244.0
8,KNN__TFIDF_NGRAMS__texto_v4_stemming,samples avg,0.8042,0.8707,0.8226,244.0



Matriz de confusión por categoría del mejor modelo calibrado:


,nombre_modelo,categoria,TN,FP,FN,TP
0,KNN__TFIDF_NGRAMS__texto_v4_stemming,cat_Pago_monto_y_calculo,153,19,4,47
1,KNN__TFIDF_NGRAMS__texto_v4_stemming,cat_Tiempos_y_oportunidad,168,4,12,39
2,KNN__TFIDF_NGRAMS__texto_v4_stemming,cat_Informacion_y_comunicacion,157,17,3,46
3,KNN__TFIDF_NGRAMS__texto_v4_stemming,cat_Atencion_y_gestion_del_servicio,155,20,6,42
4,KNN__TFIDF_NGRAMS__texto_v4_stemming,cat_Satisfaccion_o_comentario_general,164,14,9,36
